# 05 — Integração INMET × Malha Municipal IBGE

## Objetivo

Associar cada observação climática anual das estações meteorológicas do INMET ao município brasileiro correspondente.

A etapa anterior produziu dados climáticos na granularidade:

`estação meteorológica × ano`

Porém, as demais bases do projeto, como IBGE/PAM, MapBiomas, INPE e SEEG, possuem forte relacionamento com a unidade territorial municipal.

Portanto, neste notebook será realizada a transformação:

`estação × ano → município × ano`

## Estratégia

A associação será realizada através de geoprocessamento utilizando:

- latitude da estação;
- longitude da estação;
- malha municipal IBGE;
- código oficial do município (`CD_MUN`).

Cada combinação `codigo_wmo + ano` será transformada em um ponto geográfico e associada ao polígono municipal correspondente.

Essa abordagem é preferível a manter um município fixo por código WMO, pois algumas estações podem apresentar alterações de coordenadas ao longo da série histórica.

## Fonte climática

Arquivos processados:

`data/databases_processed/inmet/station_year/`

Período:

**2019–2024**

## Fonte territorial

Malha Municipal do IBGE.

Granularidade territorial:

`município`

Sistema de referência esperado:

`EPSG:4674`

## Resultado esperado

Base climática municipal:

`codigo_ibge + municipio + uf + regiao + ano`

com indicadores climáticos anuais derivados das estações INMET.

O resultado será armazenado em:

`data/databases_curated/inmet/municipio_ano/`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

In [2]:
BASE_DIR = Path(
    r"C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao"
)

PROCESSED_INMET_DIR = (
    BASE_DIR
    / "data"
    / "databases_processed"
    / "inmet"
)

STATION_YEAR_DIR = (
    PROCESSED_INMET_DIR
    / "station_year"
)

QUALITY_DIR = (
    PROCESSED_INMET_DIR
    / "quality"
)

RAW_IBGE_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "ibge_territorial"
)

CURATED_INMET_DIR = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "inmet"
)

CURATED_MUNICIPIO_ANO_DIR = (
    CURATED_INMET_DIR
    / "municipio_ano"
)

CURATED_MUNICIPIO_ANO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Station year:")
print(STATION_YEAR_DIR)

print("\nMalha IBGE:")
print(RAW_IBGE_DIR)

print("\nDestino curated:")
print(CURATED_MUNICIPIO_ANO_DIR)

Station year:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_processed\inmet\station_year

Malha IBGE:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\ibge_territorial

Destino curated:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\inmet\municipio_ano


In [3]:
arquivos_station_year = sorted(
    STATION_YEAR_DIR.glob(
        "inmet_station_year_*.csv"
    )
)

print(
    "Arquivos station_year encontrados:",
    len(arquivos_station_year)
)

for arquivo in arquivos_station_year:
    print(
        arquivo.name
    )

Arquivos station_year encontrados: 6
inmet_station_year_2019.csv
inmet_station_year_2020.csv
inmet_station_year_2021.csv
inmet_station_year_2022.csv
inmet_station_year_2023.csv
inmet_station_year_2024.csv


In [4]:
station_year_lista = []

for arquivo in arquivos_station_year:

    df = pd.read_csv(
        arquivo
    )

    station_year_lista.append(
        df
    )

station_year = pd.concat(
    station_year_lista,
    ignore_index=True
)

print(
    "Dimensão station_year:"
)

print(
    station_year.shape
)

display(
    station_year.head()
)

Dimensão station_year:
(1195, 26)


,codigo_wmo,estacao,uf,regiao,ano,latitude,longitude,altitude_m,precipitacao_anual_mm,temperatura_media_anual_c,...,dias_com_chuva,dias_precipitacao_valida,dias_observados,horas_registradas,horas_esperadas,cobertura_temporal_pct,cobertura_precipitacao_pct,cobertura_temperatura_pct,cobertura_umidade_pct,qualidade_cobertura
0,A001,BRASILIA,DF,centro_oeste,2019,-15.789343,-47.925756,1160.96,1369.4,21.964078,...,131,365,365,8760,8760,100.0,99.82,99.82,99.82,excelente
1,A042,BRAZLANDIA,DF,centro_oeste,2019,-15.599722,-48.131111,1143.00,1466.2,22.383979,...,126,363,365,8760,8760,100.0,96.61,96.62,96.62,excelente
2,A045,AGUAS EMENDADAS,DF,centro_oeste,2019,-15.596491,-47.625801,1030.36,1369.6,21.829687,...,122,365,365,8760,8760,100.0,99.43,99.44,99.44,excelente
3,A046,GAMA (PONTE ALTA),DF,centro_oeste,2019,-15.935278,-48.137500,990.00,1247.8,22.359208,...,141,365,365,8760,8760,100.0,99.99,99.99,99.99,excelente
4,A047,PARANOA (COOPA-DF),DF,centro_oeste,2019,-16.012222,-47.557417,1043.00,1187.4,22.248934,...,118,365,365,8760,8760,100.0,99.58,99.59,98.04,excelente


In [5]:
print(
    "Período:"
)

print(
    station_year["ano"].min(),
    "até",
    station_year["ano"].max()
)

print(
    "\nEstações WMO:"
)

print(
    station_year[
        "codigo_wmo"
    ].nunique()
)

print(
    "\nUFs:"
)

print(
    sorted(
        station_year[
            "uf"
        ].dropna().unique()
    )
)

print(
    "\nRegiões:"
)

print(
    station_year[
        "regiao"
    ].value_counts(
        dropna=False
    )
)

Período:
2019 até 2024

Estações WMO:
210

UFs:
['DF', 'GO', 'MS', 'MT', 'PR', 'RS', 'SC']

Regiões:
regiao
centro_oeste    631
sul             564
Name: count, dtype: int64


In [6]:
MALHA_MUNICIPAL_DIR = (
    RAW_IBGE_DIR
    / "malha_municipal"
    / "BR_Municipios_2024"
)

print("Pasta da Malha Municipal:")
print(MALHA_MUNICIPAL_DIR)

print("\nExiste?")
print(MALHA_MUNICIPAL_DIR.exists())

Pasta da Malha Municipal:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\ibge_territorial\malha_municipal\BR_Municipios_2024

Existe?
True


In [7]:
arquivos_malha = sorted(
    MALHA_MUNICIPAL_DIR.glob("*")
)

print(
    "Quantidade de arquivos:",
    len(arquivos_malha)
)

print("\nArquivos encontrados:")

for arquivo in arquivos_malha:
    print(
        arquivo.name
    )

Quantidade de arquivos: 5

Arquivos encontrados:
BR_Municipios_2024.cpg
BR_Municipios_2024.dbf
BR_Municipios_2024.prj
BR_Municipios_2024.shp
BR_Municipios_2024.shx


In [8]:
arquivos_shp = sorted(
    MALHA_MUNICIPAL_DIR.glob(
        "*.shp"
    )
)

print(
    "Shapefiles encontrados:",
    len(arquivos_shp)
)

for arquivo in arquivos_shp:
    print(
        arquivo.name
    )

Shapefiles encontrados: 1
BR_Municipios_2024.shp


In [9]:
MALHA_SHP = arquivos_shp[0]

print(
    "Shapefile utilizado:"
)

print(
    MALHA_SHP
)

Shapefile utilizado:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\raw\ibge_territorial\malha_municipal\BR_Municipios_2024\BR_Municipios_2024.shp


In [10]:
malha_municipal = gpd.read_file(
    MALHA_SHP
)

print(
    "Dimensão da malha:"
)

print(
    malha_municipal.shape
)

print(
    "\nCRS:"
)

print(
    malha_municipal.crs
)

print(
    "\nColunas:"
)

print(
    malha_municipal.columns.tolist()
)

display(
    malha_municipal.head()
)

Dimensão da malha:
(5573, 16)

CRS:
EPSG:4674

Colunas:
['CD_MUN', 'NM_MUN', 'CD_RGI', 'NM_RGI', 'CD_RGINT', 'NM_RGINT', 'CD_UF', 'NM_UF', 'SIGLA_UF', 'CD_REGIA', 'NM_REGIA', 'SIGLA_RG', 'CD_CONCU', 'NM_CONCU', 'AREA_KM2', 'geometry']


,CD_MUN,NM_MUN,CD_RGI,NM_RGI,CD_RGINT,NM_RGINT,CD_UF,NM_UF,SIGLA_UF,CD_REGIA,NM_REGIA,SIGLA_RG,CD_CONCU,NM_CONCU,AREA_KM2,geometry
0,2504108,Carrapateira,250015,Cajazeiras,2504,Sousa - Cajazeiras,25,Paraíba,PB,2,Nordeste,NE,None,None,59.070,"POLYGON ((-38.33672 -6.99279, -38.33653 -6.993..."
1,1718451,Pugmil,170003,Paraíso do Tocantins,1701,Palmas,17,Tocantins,TO,1,Norte,N,None,None,401.174,"POLYGON ((-48.91085 -10.53824, -48.911 -10.538..."
2,2104206,Fortuna,210016,Presidente Dutra,2104,Presidente Dutra,21,Maranhão,MA,2,Nordeste,NE,None,None,835.668,"POLYGON ((-43.95962 -5.49793, -43.96181 -5.497..."
3,5219902,São Francisco de Goiás,520002,Anápolis,5201,Goiânia,52,Goiás,GO,5,Centro-oeste,CO,None,None,416.535,"POLYGON ((-49.29477 -16.00852, -49.29484 -16.0..."
4,2708600,São Miguel dos Campos,270004,São Miguel dos Campos,2701,Maceió,27,Alagoas,AL,2,Nordeste,NE,None,None,335.679,"POLYGON ((-36.0739 -9.70094, -36.07339 -9.7008..."


In [12]:
malha_municipal.shape
malha_municipal.crs
malha_municipal.columns.tolist()

['CD_MUN',
 'NM_MUN',
 'CD_RGI',
 'NM_RGI',
 'CD_RGINT',
 'NM_RGINT',
 'CD_UF',
 'NM_UF',
 'SIGLA_UF',
 'CD_REGIA',
 'NM_REGIA',
 'SIGLA_RG',
 'CD_CONCU',
 'NM_CONCU',
 'AREA_KM2',
 'geometry']

In [13]:
print(
    "Quantidade de municípios:"
)

print(
    len(
        malha_municipal
    )
)

print(
    "\nGeometrias nulas:"
)

print(
    malha_municipal[
        "geometry"
    ].isna().sum()
)

print(
    "\nGeometrias vazias:"
)

print(
    malha_municipal[
        "geometry"
    ].is_empty.sum()
)

Quantidade de municípios:
5573

Geometrias nulas:
0

Geometrias vazias:
0


In [14]:
UFS_PROJETO = [
    "DF",
    "GO",
    "MS",
    "MT",
    "PR",
    "RS",
    "SC"
]

malha_projeto = (
    malha_municipal[
        malha_municipal[
            "SIGLA_UF"
        ].isin(
            UFS_PROJETO
        )
    ]
    .copy()
)

print(
    "Municípios da malha nas UFs do projeto:"
)

print(
    malha_projeto.shape
)

print(
    "\nMunicípios por UF:"
)

display(
    malha_projeto[
        "SIGLA_UF"
    ]
    .value_counts()
    .sort_index()
)

Municípios da malha nas UFs do projeto:
(1661, 16)

Municípios por UF:


SIGLA_UF
DF      1
GO    246
MS     79
MT    142
PR    399
RS    499
SC    295
Name: count, dtype: int64

In [15]:
print(
    sorted(
        malha_projeto[
            "SIGLA_UF"
        ].unique()
    )
)

['DF', 'GO', 'MS', 'MT', 'PR', 'RS', 'SC']


In [16]:
malha_join = (
    malha_projeto[
        [
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "NM_REGIA",
            "AREA_KM2",
            "geometry"
        ]
    ]
    .copy()
)

print(
    malha_join.shape
)

display(
    malha_join.head()
)

(1661, 6)


,CD_MUN,NM_MUN,SIGLA_UF,NM_REGIA,AREA_KM2,geometry
3,5219902,São Francisco de Goiás,GO,Centro-oeste,416.535,"POLYGON ((-49.29477 -16.00852, -49.29484 -16.0..."
7,4316204,Rondinha,RS,Sul,252.454,"POLYGON ((-52.91535 -27.88582, -52.91559 -27.8..."
8,4317558,Santo Antônio do Palma,RS,Sul,126.094,"POLYGON ((-51.99835 -28.43782, -51.99822 -28.4..."
9,4209508,Laurentino,SC,Sul,79.333,"POLYGON ((-49.7267 -27.14486, -49.72657 -27.14..."
12,4202107,Barra Velha,SC,Sul,138.947,"MULTIPOLYGON (((-48.6184 -26.71031, -48.61851 ..."


In [17]:
malha_join[
    "CD_MUN"
] = (
    malha_join[
        "CD_MUN"
    ]
    .astype(str)
    .str.strip()
)

print(
    malha_join[
        "CD_MUN"
    ].dtype
)

print(
    malha_join[
        "CD_MUN"
    ].str.len().value_counts()
)

object
CD_MUN
7    1661
Name: count, dtype: int64


In [18]:
print(
    "Latitude ausente:"
)

print(
    station_year[
        "latitude"
    ].isna().sum()
)

print(
    "\nLongitude ausente:"
)

print(
    station_year[
        "longitude"
    ].isna().sum()
)

print(
    "\nLatitude fora do intervalo:"
)

print(
    (
        ~station_year[
            "latitude"
        ].between(
            -90,
            90
        )
    ).sum()
)

print(
    "\nLongitude fora do intervalo:"
)

print(
    (
        ~station_year[
            "longitude"
        ].between(
            -180,
            180
        )
    ).sum()
)

Latitude ausente:
0

Longitude ausente:
0

Latitude fora do intervalo:
0

Longitude fora do intervalo:
0


In [19]:
station_year_geo = (
    gpd.GeoDataFrame(
        station_year.copy(),
        geometry=gpd.points_from_xy(
            station_year[
                "longitude"
            ],
            station_year[
                "latitude"
            ]
        ),
        crs="EPSG:4674"
    )
)

print(
    "CRS INMET:"
)

print(
    station_year_geo.crs
)

print(
    "\nCRS Malha:"
)

print(
    malha_join.crs
)

print(
    "\nQuantidade de pontos:"
)

print(
    len(
        station_year_geo
    )
)

display(
    station_year_geo[
        [
            "codigo_wmo",
            "estacao",
            "uf",
            "ano",
            "latitude",
            "longitude",
            "geometry"
        ]
    ].head()
)

CRS INMET:
EPSG:4674

CRS Malha:
EPSG:4674

Quantidade de pontos:
1195


,codigo_wmo,estacao,uf,ano,latitude,longitude,geometry
0,A001,BRASILIA,DF,2019,-15.789343,-47.925756,POINT (-47.92576 -15.78934)
1,A042,BRAZLANDIA,DF,2019,-15.599722,-48.131111,POINT (-48.13111 -15.59972)
2,A045,AGUAS EMENDADAS,DF,2019,-15.596491,-47.625801,POINT (-47.6258 -15.59649)
3,A046,GAMA (PONTE ALTA),DF,2019,-15.935278,-48.137500,POINT (-48.1375 -15.93528)
4,A047,PARANOA (COOPA-DF),DF,2019,-16.012222,-47.557417,POINT (-47.55742 -16.01222)


In [20]:
station_year_municipio = (
    gpd.sjoin(
        station_year_geo,
        malha_join,
        how="left",
        predicate="within"
    )
)

print(
    "Dimensão após spatial join:"
)

print(
    station_year_municipio.shape
)

display(
    station_year_municipio[
        [
            "codigo_wmo",
            "estacao",
            "ano",
            "uf",
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "NM_REGIA"
        ]
    ].head(20)
)

Dimensão após spatial join:
(1195, 33)


,codigo_wmo,estacao,ano,uf,CD_MUN,NM_MUN,SIGLA_UF,NM_REGIA
0,A001,BRASILIA,2019,DF,5300108,Brasília,DF,Centro-oeste
1,A042,BRAZLANDIA,2019,DF,5300108,Brasília,DF,Centro-oeste
2,A045,AGUAS EMENDADAS,2019,DF,5300108,Brasília,DF,Centro-oeste
3,A046,GAMA (PONTE ALTA),2019,DF,5300108,Brasília,DF,Centro-oeste
4,A047,PARANOA (COOPA-DF),2019,DF,5300108,Brasília,DF,Centro-oeste
5,A002,GOIANIA,2019,GO,5208707,Goiânia,GO,Centro-oeste
6,A003,MORRINHOS,2019,GO,5213806,Morrinhos,GO,Centro-oeste
7,A005,PORANGATU,2019,GO,5218003,Porangatu,GO,Centro-oeste
8,A011,SAO SIMAO,2019,GO,5220405,São Simão,GO,Centro-oeste
9,A012,LUZIANIA,2019,GO,5212501,Luziânia,GO,Centro-oeste


In [21]:
print(
    "Total de estação-ano:"
)

print(
    len(
        station_year_municipio
    )
)

print(
    "\nSem município associado:"
)

print(
    station_year_municipio[
        "CD_MUN"
    ].isna().sum()
)

print(
    "\nCom município associado:"
)

print(
    station_year_municipio[
        "CD_MUN"
    ].notna().sum()
)

print(
    "\nMunicípios diferentes atingidos:"
)

print(
    station_year_municipio[
        "CD_MUN"
    ].nunique()
)

Total de estação-ano:
1195

Sem município associado:
0

Com município associado:
1195

Municípios diferentes atingidos:
206


In [22]:
duplicados_join = (
    station_year_municipio[
        [
            "codigo_wmo",
            "ano"
        ]
    ]
    .duplicated(
        keep=False
    )
)

print(
    "Registros duplicados após spatial join:"
)

print(
    duplicados_join.sum()
)

Registros duplicados após spatial join:
0


In [23]:
print(
    "Combinações codigo_wmo + ano:"
)

print(
    station_year_municipio[
        [
            "codigo_wmo",
            "ano"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

Combinações codigo_wmo + ano:
1195


In [24]:
station_year_municipio[
    "uf_confere"
] = (
    station_year_municipio[
        "uf"
    ]
    ==
    station_year_municipio[
        "SIGLA_UF"
    ]
)

print(
    "UF INMET diferente da UF IBGE:"
)

print(
    (
        ~station_year_municipio[
            "uf_confere"
        ]
    ).sum()
)

UF INMET diferente da UF IBGE:
1


In [25]:
display(
    station_year_municipio[
        ~station_year_municipio[
            "uf_confere"
        ]
    ][
        [
            "codigo_wmo",
            "estacao",
            "ano",
            "uf",
            "latitude",
            "longitude",
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF"
        ]
    ]
)

,codigo_wmo,estacao,ano,uf,latitude,longitude,CD_MUN,NM_MUN,SIGLA_UF
207,A895,CHAPECO,2019,SC,-27.955278,-52.635556,4314779,Pontão,RS


In [26]:
display(
    station_year_municipio[
        [
            "codigo_wmo",
            "estacao",
            "ano",
            "uf",
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "NM_REGIA"
        ]
    ]
    .sort_values(
        [
            "uf",
            "NM_MUN",
            "codigo_wmo",
            "ano"
        ]
    )
    .head(50)
)

,codigo_wmo,estacao,ano,uf,CD_MUN,NM_MUN,SIGLA_UF,NM_REGIA
0,A001,BRASILIA,2019,DF,5300108,Brasília,DF,Centro-oeste
209,A001,BRASILIA,2020,DF,5300108,Brasília,DF,Centro-oeste
417,A001,BRASILIA,2021,DF,5300108,Brasília,DF,Centro-oeste
624,A001,BRASILIA,2022,DF,5300108,Brasília,DF,Centro-oeste
814,A001,BRASILIA,2023,DF,5300108,Brasília,DF,Centro-oeste
1005,A001,BRASILIA,2024,DF,5300108,Brasília,DF,Centro-oeste
1,A042,BRAZLANDIA,2019,DF,5300108,Brasília,DF,Centro-oeste
210,A042,BRAZLANDIA,2020,DF,5300108,Brasília,DF,Centro-oeste
418,A042,BRAZLANDIA,2021,DF,5300108,Brasília,DF,Centro-oeste
625,A042,BRAZLANDIA,2022,DF,5300108,Brasília,DF,Centro-oeste


## 5.4 Auditoria geográfica das estações INMET

O spatial join entre as coordenadas das estações meteorológicas do INMET e a
Malha Municipal 2024 do IBGE preservou integralmente a granularidade da base:

- 1.195 registros estação × ano antes do cruzamento;
- 1.195 registros após o cruzamento;
- nenhuma estação sem município associado;
- nenhuma duplicidade de `codigo_wmo + ano`;
- 206 municípios distintos associados às estações.

Foi identificada uma única divergência entre a UF declarada nos metadados do
INMET e a UF obtida espacialmente pela Malha Municipal do IBGE:

- estação A895 — CHAPECO — ano 2019;
- UF INMET: SC;
- município obtido pelas coordenadas: Pontão/RS.

O caso será auditado separadamente antes da geração da camada CURATED.
Os dados RAW e PROCESSED não serão modificados; eventuais ajustes geográficos
serão documentados e realizados somente na camada de integração/curadoria.

In [27]:
a895_historico = (
    station_year_municipio[
        station_year_municipio[
            "codigo_wmo"
        ] == "A895"
    ][
        [
            "codigo_wmo",
            "estacao",
            "ano",
            "uf",
            "latitude",
            "longitude",
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "uf_confere"
        ]
    ]
    .sort_values(
        "ano"
    )
)

display(
    a895_historico
)

,codigo_wmo,estacao,ano,uf,latitude,longitude,CD_MUN,NM_MUN,SIGLA_UF,uf_confere
207,A895,CHAPECO,2019,SC,-27.955278,-52.635556,4314779,Pontão,RS,False
415,A895,CHAPECO,2020,SC,-27.085311,-52.635711,4204202,Chapecó,SC,True
622,A895,CHAPECO,2021,SC,-27.085311,-52.635711,4204202,Chapecó,SC,True
812,A895,CHAPECO,2022,SC,-27.085311,-52.635711,4204202,Chapecó,SC,True
1003,A895,CHAPECO,2023,SC,-27.085311,-52.635711,4204202,Chapecó,SC,True
1193,A895,CHAPECO,2024,SC,-27.085311,-52.635711,4204202,Chapecó,SC,True


In [28]:
display(
    station_year[
        station_year[
            "codigo_wmo"
        ] == "A895"
    ][
        [
            "ano",
            "codigo_wmo",
            "estacao",
            "uf",
            "latitude",
            "longitude"
        ]
    ]
    .sort_values(
        "ano"
    )
)

,ano,codigo_wmo,estacao,uf,latitude,longitude
207,2019,A895,CHAPECO,SC,-27.955278,-52.635556
415,2020,A895,CHAPECO,SC,-27.085311,-52.635711
622,2021,A895,CHAPECO,SC,-27.085311,-52.635711
812,2022,A895,CHAPECO,SC,-27.085311,-52.635711
1003,2023,A895,CHAPECO,SC,-27.085311,-52.635711
1193,2024,A895,CHAPECO,SC,-27.085311,-52.635711


In [29]:
a895_coords = (
    station_year[
        station_year[
            "codigo_wmo"
        ] == "A895"
    ][
        [
            "ano",
            "latitude",
            "longitude"
        ]
    ]
    .sort_values(
        "ano"
    )
    .reset_index(
        drop=True
    )
)

a895_coords[
    "delta_latitude"
] = (
    a895_coords[
        "latitude"
    ]
    .diff()
)

a895_coords[
    "delta_longitude"
] = (
    a895_coords[
        "longitude"
    ]
    .diff()
)

display(
    a895_coords
)

,ano,latitude,longitude,delta_latitude,delta_longitude
0,2019,-27.955278,-52.635556,NaN,NaN
1,2020,-27.085311,-52.635711,0.869967,-0.000156
2,2021,-27.085311,-52.635711,0.000000,0.000000
3,2022,-27.085311,-52.635711,0.000000,0.000000
4,2023,-27.085311,-52.635711,0.000000,0.000000
5,2024,-27.085311,-52.635711,0.000000,0.000000


In [30]:
municipios_por_estacao = (
    station_year_municipio
    .groupby(
        "codigo_wmo"
    )
    .agg(
        anos=(
            "ano",
            "nunique"
        ),

        municipios=(
            "CD_MUN",
            "nunique"
        ),

        ufs_ibge=(
            "SIGLA_UF",
            "nunique"
        )
    )
    .reset_index()
)

display(
    municipios_por_estacao
    .sort_values(
        [
            "ufs_ibge",
            "municipios"
        ],
        ascending=False
    )
    .head(30)
)

,codigo_wmo,anos,municipios,ufs_ibge
146,A895,6,2,2
114,A857,6,2,1
160,A911,6,2,1
0,A001,6,1,1
1,A002,6,1,1
2,A003,6,1,1
3,A005,6,1,1
4,A011,6,1,1
5,A012,6,1,1
6,A013,6,1,1


In [31]:
estacoes_mudaram_municipio = (
    municipios_por_estacao[
        municipios_por_estacao[
            "municipios"
        ] > 1
    ]
    .copy()
)

print(
    "Estações associadas a mais de um município ao longo dos anos:"
)

print(
    len(
        estacoes_mudaram_municipio
    )
)

display(
    estacoes_mudaram_municipio
)

Estações associadas a mais de um município ao longo dos anos:
3


,codigo_wmo,anos,municipios,ufs_ibge
114,A857,6,2,1
146,A895,6,2,2
160,A911,6,2,1


In [32]:
detalhes_mudanca_municipio = (
    station_year_municipio[
        station_year_municipio[
            "codigo_wmo"
        ].isin(
            estacoes_mudaram_municipio[
                "codigo_wmo"
            ]
        )
    ][
        [
            "codigo_wmo",
            "estacao",
            "ano",
            "uf",
            "latitude",
            "longitude",
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF"
        ]
    ]
    .sort_values(
        [
            "codigo_wmo",
            "ano"
        ]
    )
)

display(
    detalhes_mudanca_municipio
)

,codigo_wmo,estacao,ano,uf,latitude,longitude,CD_MUN,NM_MUN,SIGLA_UF
194,A857,SAO MIGUEL DO OESTE,2019,SC,-26.776667,-53.504444,4204905,Descanso,SC
402,A857,SAO MIGUEL DO OESTE,2020,SC,-26.786111,-53.513889,4217204,São Miguel do Oeste,SC
609,A857,SAO MIGUEL DO OESTE,2021,SC,-26.786111,-53.513889,4217204,São Miguel do Oeste,SC
799,A857,SAO MIGUEL DO OESTE,2022,SC,-26.776389,-53.504167,4204905,Descanso,SC
990,A857,SAO MIGUEL DO OESTE,2023,SC,-26.786389,-53.514167,4217204,São Miguel do Oeste,SC
1180,A857,SAO MIGUEL DO OESTE,2024,SC,-26.786389,-53.514167,4217204,São Miguel do Oeste,SC
207,A895,CHAPECO,2019,SC,-27.955278,-52.635556,4314779,Pontão,RS
415,A895,CHAPECO,2020,SC,-27.085311,-52.635711,4204202,Chapecó,SC
622,A895,CHAPECO,2021,SC,-27.085311,-52.635711,4204202,Chapecó,SC
812,A895,CHAPECO,2022,SC,-27.085311,-52.635711,4204202,Chapecó,SC


In [33]:
station_year_municipio[
    ~station_year_municipio[
        "uf_confere"
    ]
].to_csv(
    QUALITY_DIR
    / "divergencias_uf_spatial_join_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

detalhes_mudanca_municipio.to_csv(
    QUALITY_DIR
    / "estacoes_mudanca_municipio_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Auditorias geográficas salvas."
)

Auditorias geográficas salvas.


## 5.5 — Curadoria das inconsistências geográficas

A auditoria espacial identificou três estações associadas a mais de um município ao longo da série histórica.

### A857 — São Miguel do Oeste/SC

A estação permaneceu em Santa Catarina, alternando entre os municípios de
São Miguel do Oeste e Descanso devido às diferentes coordenadas anuais.

Como não foi identificada divergência de UF e as posições permanecem próximas,
as coordenadas anuais serão preservadas.

### A895 — Chapecó/SC

Em 2019, a coordenada registrada pelo INMET foi associada espacialmente ao
município de Pontão/RS, apesar de a estação estar identificada como Chapecó/SC.

De 2020 a 2024 a estação apresenta coordenadas estáveis em Chapecó/SC.

O registro de 2019 será preservado nas camadas RAW e PROCESSED, porém não será
utilizado como referência espacial na camada CURATED.

### A911 — Sapezal/MT

Em 2019, a coordenada registrada foi associada ao município de Aripuanã/MT.

De 2020 a 2024 a estação apresenta coordenadas estáveis em Sapezal/MT.

O registro de 2019 será preservado nas camadas RAW e PROCESSED, porém não será
utilizado como referência espacial na camada CURATED.

Nenhum dado original será sobrescrito.

In [34]:
station_year_auditado = (
    station_year_municipio
    .copy()
)

station_year_auditado[
    "qualidade_geografica"
] = "ok"

station_year_auditado[
    "observacao_geografica"
] = pd.NA

station_year_auditado[
    "usar_na_curated"
] = True

In [35]:
station_year_auditado = (
    station_year_municipio
    .copy()
)

station_year_auditado[
    "qualidade_geografica"
] = "ok"

station_year_auditado[
    "observacao_geografica"
] = pd.NA

station_year_auditado[
    "usar_na_curated"
] = True

In [36]:
mascara_a895_2019 = (
    (
        station_year_auditado[
            "codigo_wmo"
        ] == "A895"
    )
    &
    (
        station_year_auditado[
            "ano"
        ] == 2019
    )
)


mascara_a911_2019 = (
    (
        station_year_auditado[
            "codigo_wmo"
        ] == "A911"
    )
    &
    (
        station_year_auditado[
            "ano"
        ] == 2019
    )
)

In [37]:
station_year_auditado.loc[
    mascara_a895_2019,
    "qualidade_geografica"
] = "inconsistente"

station_year_auditado.loc[
    mascara_a895_2019,
    "observacao_geografica"
] = (
    "UF INMET SC; coordenada espacial em Pontao/RS"
)

station_year_auditado.loc[
    mascara_a895_2019,
    "usar_na_curated"
] = False


station_year_auditado.loc[
    mascara_a911_2019,
    "qualidade_geografica"
] = "inconsistente"

station_year_auditado.loc[
    mascara_a911_2019,
    "observacao_geografica"
] = (
    "estacao Sapezal; coordenada 2019 espacialmente em Aripuana/MT; "
    "2020-2024 estavel em Sapezal"
)

station_year_auditado.loc[
    mascara_a911_2019,
    "usar_na_curated"
] = False

In [39]:
print(
    "Qualidade geográfica:"
)

print(
    station_year_auditado[
        "qualidade_geografica"
    ].value_counts()
)


print(
    "\nUsados na CURATED:"
)

print(
    station_year_auditado[
        "usar_na_curated"
    ].value_counts()
)

Qualidade geográfica:
qualidade_geografica
ok               1193
inconsistente       2
Name: count, dtype: int64

Usados na CURATED:
usar_na_curated
True     1193
False       2
Name: count, dtype: int64


In [40]:
display(
    station_year_auditado[
        ~station_year_auditado[
            "usar_na_curated"
        ]
    ][
        [
            "codigo_wmo",
            "estacao",
            "ano",
            "uf",
            "latitude",
            "longitude",
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "qualidade_geografica",
            "observacao_geografica"
        ]
    ]
)

,codigo_wmo,estacao,ano,uf,latitude,longitude,CD_MUN,NM_MUN,SIGLA_UF,qualidade_geografica,observacao_geografica
86,A911,SAPEZAL,2019,MT,-10.165833,-59.451111,5101407,Aripuanã,MT,inconsistente,estacao Sapezal; coordenada 2019 espacialmente...
207,A895,CHAPECO,2019,SC,-27.955278,-52.635556,4314779,Pontão,RS,inconsistente,UF INMET SC; coordenada espacial em Pontao/RS


In [41]:
station_year_auditado.drop(
    columns=[
        "geometry",
        "index_right"
    ],
    errors="ignore"
).to_csv(
    QUALITY_DIR
    / "station_year_geocodificado_auditado_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Base geográfica auditada salva."
)

Base geográfica auditada salva.


In [42]:
estacoes_validas = (
    station_year_auditado[
        station_year_auditado[
            "usar_na_curated"
        ]
    ]
    .copy()
)

print(
    "Estação-ano válidas:"
)

print(
    len(
        estacoes_validas
    )
)

Estação-ano válidas:
1193


In [43]:
estacoes_validas[
    "codigo_ibge"
] = (
    estacoes_validas[
        "CD_MUN"
    ]
    .astype(str)
    .str.strip()
)

estacoes_validas[
    "municipio"
] = (
    estacoes_validas[
        "NM_MUN"
    ]
)

estacoes_validas[
    "uf_municipio"
] = (
    estacoes_validas[
        "SIGLA_UF"
    ]
)

estacoes_validas[
    "regiao_municipio"
] = (
    estacoes_validas[
        "NM_REGIA"
    ]
    .str.lower()
    .str.replace(
        "-",
        "_",
        regex=False
    )
)

In [44]:
display(
    estacoes_validas[
        [
            "codigo_wmo",
            "ano",
            "codigo_ibge",
            "municipio",
            "uf_municipio",
            "regiao_municipio",
            "precipitacao_anual_mm",
            "temperatura_media_anual_c"
        ]
    ].head()
)

,codigo_wmo,ano,codigo_ibge,municipio,uf_municipio,regiao_municipio,precipitacao_anual_mm,temperatura_media_anual_c
0,A001,2019,5300108,Brasília,DF,centro_oeste,1369.4,21.964078
1,A042,2019,5300108,Brasília,DF,centro_oeste,1466.2,22.383979
2,A045,2019,5300108,Brasília,DF,centro_oeste,1369.6,21.829687
3,A046,2019,5300108,Brasília,DF,centro_oeste,1247.8,22.359208
4,A047,2019,5300108,Brasília,DF,centro_oeste,1187.4,22.248934


In [45]:
COBERTURA_MINIMA = 80.0

In [46]:
def media_ponderada_cobertura(
    grupo,
    coluna_valor,
    coluna_cobertura,
    cobertura_minima=80.0
):

    mascara = (
        grupo[
            coluna_valor
        ].notna()
        &
        grupo[
            coluna_cobertura
        ].notna()
        &
        (
            grupo[
                coluna_cobertura
            ] >= cobertura_minima
        )
    )

    if not mascara.any():
        return np.nan

    valores = grupo.loc[
        mascara,
        coluna_valor
    ]

    pesos = grupo.loc[
        mascara,
        coluna_cobertura
    ]

    return np.average(
        valores,
        weights=pesos
    )

In [47]:
def media_ou_nan(serie):

    serie = serie.dropna()

    if len(serie) == 0:
        return np.nan

    return serie.mean()


def max_ou_nan(serie):

    serie = serie.dropna()

    if len(serie) == 0:
        return np.nan

    return serie.max()


def min_ou_nan(serie):

    serie = serie.dropna()

    if len(serie) == 0:
        return np.nan

    return serie.min()

In [48]:
registros_municipio_ano = []

grupos = (
    estacoes_validas
    .groupby(
        [
            "codigo_ibge",
            "municipio",
            "uf_municipio",
            "regiao_municipio",
            "ano"
        ],
        dropna=False
    )
)

for (
    codigo_ibge,
    municipio,
    uf,
    regiao,
    ano
), grupo in grupos:

    precip_ok = (
        grupo[
            "cobertura_precipitacao_pct"
        ] >= COBERTURA_MINIMA
    )

    temp_ok = (
        grupo[
            "cobertura_temperatura_pct"
        ] >= COBERTURA_MINIMA
    )

    umidade_ok = (
        grupo[
            "cobertura_umidade_pct"
        ] >= COBERTURA_MINIMA
    )

    grupo_temp = grupo[
        temp_ok
    ]

    registros_municipio_ano.append(
        {
            "codigo_ibge":
                codigo_ibge,

            "municipio":
                municipio,

            "uf":
                uf,

            "regiao":
                regiao,

            "ano":
                int(ano),

            "numero_estacoes":
                int(
                    grupo[
                        "codigo_wmo"
                    ].nunique()
                ),

            "numero_estacoes_precipitacao_validas":
                int(
                    grupo.loc[
                        precip_ok,
                        "codigo_wmo"
                    ].nunique()
                ),

            "numero_estacoes_temperatura_validas":
                int(
                    grupo.loc[
                        temp_ok,
                        "codigo_wmo"
                    ].nunique()
                ),

            "numero_estacoes_umidade_validas":
                int(
                    grupo.loc[
                        umidade_ok,
                        "codigo_wmo"
                    ].nunique()
                ),

            "precipitacao_anual_mm":
                media_ponderada_cobertura(
                    grupo,
                    "precipitacao_anual_mm",
                    "cobertura_precipitacao_pct",
                    COBERTURA_MINIMA
                ),

            "temperatura_media_anual_c":
                media_ponderada_cobertura(
                    grupo,
                    "temperatura_media_anual_c",
                    "cobertura_temperatura_pct",
                    COBERTURA_MINIMA
                ),

            "temperatura_maxima_anual_c":
                max_ou_nan(
                    grupo_temp[
                        "temperatura_maxima_anual_c"
                    ]
                ),

            "temperatura_minima_anual_c":
                min_ou_nan(
                    grupo_temp[
                        "temperatura_minima_anual_c"
                    ]
                ),

            "umidade_media_anual_pct":
                media_ponderada_cobertura(
                    grupo,
                    "umidade_media_anual_pct",
                    "cobertura_umidade_pct",
                    COBERTURA_MINIMA
                ),

            "radiacao_anual_kj_m2":
                media_ou_nan(
                    grupo[
                        "radiacao_anual_kj_m2"
                    ]
                ),

            "rajada_maxima_anual_ms":
                max_ou_nan(
                    grupo[
                        "rajada_maxima_anual_ms"
                    ]
                ),

            "velocidade_vento_media_anual_ms":
                media_ou_nan(
                    grupo[
                        "velocidade_vento_media_anual_ms"
                    ]
                ),

            "dias_com_chuva":
                media_ponderada_cobertura(
                    grupo,
                    "dias_com_chuva",
                    "cobertura_precipitacao_pct",
                    COBERTURA_MINIMA
                ),

            "cobertura_temporal_media_pct":
                media_ou_nan(
                    grupo[
                        "cobertura_temporal_pct"
                    ]
                ),

            "cobertura_precipitacao_media_pct":
                media_ou_nan(
                    grupo[
                        "cobertura_precipitacao_pct"
                    ]
                ),

            "cobertura_temperatura_media_pct":
                media_ou_nan(
                    grupo[
                        "cobertura_temperatura_pct"
                    ]
                ),

            "cobertura_umidade_media_pct":
                media_ou_nan(
                    grupo[
                        "cobertura_umidade_pct"
                    ]
                ),

            "origem_clima":
                "observado_inmet"
        }
    )

In [49]:
inmet_municipio_ano_observado = (
    pd.DataFrame(
        registros_municipio_ano
    )
)

print(
    "Dimensão município × ano observado:"
)

print(
    inmet_municipio_ano_observado.shape
)

display(
    inmet_municipio_ano_observado.head()
)

Dimensão município × ano observado:
(1154, 23)


,codigo_ibge,municipio,uf,regiao,ano,numero_estacoes,numero_estacoes_precipitacao_validas,numero_estacoes_temperatura_validas,numero_estacoes_umidade_validas,precipitacao_anual_mm,...,umidade_media_anual_pct,radiacao_anual_kj_m2,rajada_maxima_anual_ms,velocidade_vento_media_anual_ms,dias_com_chuva,cobertura_temporal_media_pct,cobertura_precipitacao_media_pct,cobertura_temperatura_media_pct,cobertura_umidade_media_pct,origem_clima
0,4103909,Campina da Lagoa,PR,sul,2019,1,0,0,0,NaN,...,NaN,5234152.0,30.1,3.075135,NaN,100.0,76.37,76.36,76.36,observado_inmet
1,4103909,Campina da Lagoa,PR,sul,2020,1,0,0,0,NaN,...,NaN,2239797.7,21.4,3.055536,NaN,100.0,13.14,13.11,13.11,observado_inmet
2,4103909,Campina da Lagoa,PR,sul,2021,1,0,0,0,NaN,...,NaN,1669954.4,15.7,2.980980,NaN,100.0,17.45,17.47,17.47,observado_inmet
3,4103909,Campina da Lagoa,PR,sul,2022,1,0,0,0,NaN,...,NaN,6323646.9,21.2,3.150053,NaN,100.0,65.76,65.74,65.74,observado_inmet
4,4103909,Campina da Lagoa,PR,sul,2023,1,1,1,1,1208.2,...,76.219239,5947468.1,21.9,3.159699,99.0,100.0,81.64,81.64,81.64,observado_inmet


In [50]:
print(
    "Duplicidades codigo_ibge + ano:"
)

print(
    inmet_municipio_ano_observado[
        [
            "codigo_ibge",
            "ano"
        ]
    ]
    .duplicated()
    .sum()
)

Duplicidades codigo_ibge + ano:
0


In [51]:
print(
    "Municípios diferentes:"
)

print(
    inmet_municipio_ano_observado[
        "codigo_ibge"
    ].nunique()
)

Municípios diferentes:
204


In [52]:
display(
    inmet_municipio_ano_observado[
        "ano"
    ]
    .value_counts()
    .sort_index()
)

ano
2019    201
2020    202
2021    201
2022    183
2023    184
2024    183
Name: count, dtype: int64

In [53]:
display(
    inmet_municipio_ano_observado[
        "numero_estacoes"
    ]
    .describe()
)

count    1154.000000
mean        1.033795
std         0.308427
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         5.000000
Name: numero_estacoes, dtype: float64

In [54]:
display(
    inmet_municipio_ano_observado[
        [
            "municipio",
            "uf",
            "ano",
            "numero_estacoes"
        ]
    ]
    .sort_values(
        "numero_estacoes",
        ascending=False
    )
    .head(20)
)

,municipio,uf,ano,numero_estacoes
1153,Brasília,DF,2024,5
1152,Brasília,DF,2023,5
1151,Brasília,DF,2022,5
1150,Brasília,DF,2021,5
1149,Brasília,DF,2020,5
1148,Brasília,DF,2019,5
1027,Cristalina,GO,2024,2
634,Corumbá,MS,2020,2
635,Corumbá,MS,2021,2
636,Corumbá,MS,2022,2


In [55]:
ARQUIVO_OBSERVADO = (
    CURATED_MUNICIPIO_ANO_DIR
    / "inmet_municipio_ano_observado_2019_2024.csv"
)

inmet_municipio_ano_observado.to_csv(
    ARQUIVO_OBSERVADO,
    index=False,
    encoding="utf-8-sig"
)

print(
    "CURATED observada salva em:"
)

print(
    ARQUIVO_OBSERVADO
)

CURATED observada salva em:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\inmet\municipio_ano\inmet_municipio_ano_observado_2019_2024.csv


In [56]:
inmet_municipio_ano_observado[
    "status_dados_climaticos"
] = np.select(
    [
        (
            (
                inmet_municipio_ano_observado[
                    "numero_estacoes_precipitacao_validas"
                ] > 0
            )
            &
            (
                inmet_municipio_ano_observado[
                    "numero_estacoes_temperatura_validas"
                ] > 0
            )
            &
            (
                inmet_municipio_ano_observado[
                    "numero_estacoes_umidade_validas"
                ] > 0
            )
        ),

        (
            (
                inmet_municipio_ano_observado[
                    "numero_estacoes_precipitacao_validas"
                ] > 0
            )
            |
            (
                inmet_municipio_ano_observado[
                    "numero_estacoes_temperatura_validas"
                ] > 0
            )
            |
            (
                inmet_municipio_ano_observado[
                    "numero_estacoes_umidade_validas"
                ] > 0
            )
        )
    ],

    [
        "observado_completo",
        "observado_parcial"
    ],

    default="sem_variaveis_principais_validas"
)

In [58]:
display(
    inmet_municipio_ano_observado[
        "status_dados_climaticos"
    ]
    .value_counts()
)

status_dados_climaticos
observado_completo                  629
sem_variaveis_principais_validas    376
observado_parcial                   149
Name: count, dtype: int64

In [64]:
# Atualizando a coluna de origem para uma nomenclatura mais precisa
inmet_municipio_ano_observado["origem_clima"] = "estacao_no_municipio"

# Validação rápida da alteração
print("Nova distribuição de origem_clima:")
display(
    inmet_municipio_ano_observado["origem_clima"]
    .value_counts()
)

Nova distribuição de origem_clima:


origem_clima
estacao_no_municipio    1154
Name: count, dtype: int64

In [65]:
colunas_clima_principais = [
    "precipitacao_anual_mm",
    "temperatura_media_anual_c",
    "umidade_media_anual_pct"
]

print(
    "Valores ausentes nas variáveis principais:"
)

display(
    inmet_municipio_ano_observado[
        colunas_clima_principais
    ]
    .isna()
    .sum()
)

Valores ausentes nas variáveis principais:


precipitacao_anual_mm        470
temperatura_media_anual_c    379
umidade_media_anual_pct      438
dtype: int64

In [66]:
display(
    inmet_municipio_ano_observado[
        colunas_clima_principais
    ]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .rename(
        "percentual_ausente"
    )
)

precipitacao_anual_mm        40.73
temperatura_media_anual_c    32.84
umidade_media_anual_pct      37.95
Name: percentual_ausente, dtype: float64

In [67]:
display(
    inmet_municipio_ano_observado[
        [
            "cobertura_precipitacao_media_pct",
            "cobertura_temperatura_media_pct",
            "cobertura_umidade_media_pct"
        ]
    ]
    .describe()
)

,cobertura_precipitacao_media_pct,cobertura_temperatura_media_pct,cobertura_umidade_media_pct
count,1154.000000,1154.000000,1154.000000
mean,72.407417,78.417935,75.143172
std,33.058375,30.102094,31.236037
min,0.000000,0.000000,0.000000
25%,51.505000,67.010000,59.980000
50%,88.255000,94.450000,90.245000
75%,99.210000,99.757500,99.235000
max,100.000000,100.000000,100.000000


In [68]:
inmet_municipio_ano_observado.to_csv(
    ARQUIVO_OBSERVADO,
    index=False,
    encoding="utf-8-sig"
)

print(
    "CURATED observada atualizada:"
)

print(
    ARQUIVO_OBSERVADO
)

CURATED observada atualizada:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\inmet\municipio_ano\inmet_municipio_ano_observado_2019_2024.csv


## 5.6 — Preparação espacial para estimativa climática municipal

A rede do INMET não possui estação meteorológica em todos os municípios.

Para permitir a atribuição de indicadores climáticos aos municípios sem
observação direta, será avaliada uma metodologia de proximidade espacial.

As distâncias não serão calculadas no CRS geográfico EPSG:4674, pois suas
unidades são graus.

Para os cálculos espaciais será utilizada temporariamente uma projeção métrica,
mantendo a Malha Municipal original em EPSG:4674.

In [69]:
CRS_METRICO = "EPSG:5880"

malha_metrica = (
    malha_join
    .to_crs(
        CRS_METRICO
    )
    .copy()
)

print(
    "CRS da malha métrica:"
)

print(
    malha_metrica.crs
)

CRS da malha métrica:
EPSG:5880


In [70]:
municipios_pontos = (
    malha_metrica[
        [
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "NM_REGIA",
            "AREA_KM2",
            "geometry"
        ]
    ]
    .copy()
)

municipios_pontos[
    "geometry"
] = (
    municipios_pontos[
        "geometry"
    ]
    .centroid
)

print(
    "Municípios:"
)

print(
    len(
        municipios_pontos
    )
)

print(
    "\nCRS:"
)

print(
    municipios_pontos.crs
)

display(
    municipios_pontos.head()
)

Municípios:
1661

CRS:
EPSG:5880


,CD_MUN,NM_MUN,SIGLA_UF,NM_REGIA,AREA_KM2,geometry
3,5219902,São Francisco de Goiás,GO,Centro-oeste,416.535,POINT (5507990.155 8229947.355)
7,4316204,Rondinha,RS,Sul,252.454,POINT (5109043.663 6919630.857)
8,4317558,Santo Antônio do Palma,RS,Sul,126.094,POINT (5194962.085 6845017.066)
9,4209508,Laurentino,SC,Sul,79.333,POINT (5422596.434 6982130.37)
12,4202107,Barra Velha,SC,Sul,138.947,POINT (5524738.105 7038937.163)


In [71]:
estacoes_geo_validas = gpd.GeoDataFrame(
    estacoes_validas.drop(
        columns="geometry",
        errors="ignore"
    ).copy(),

    geometry=gpd.points_from_xy(
        estacoes_validas[
            "longitude"
        ],

        estacoes_validas[
            "latitude"
        ]
    ),

    crs="EPSG:4674"
)

In [72]:
estacoes_metrica = (
    estacoes_geo_validas
    .to_crs(
        CRS_METRICO
    )
)

print(
    "Quantidade estação-ano:"
)

print(
    len(
        estacoes_metrica
    )
)

print(
    "\nCRS:"
)

print(
    estacoes_metrica.crs
)

Quantidade estação-ano:
1193

CRS:
EPSG:5880


In [74]:
estacoes_precipitacao_validas = (
    estacoes_metrica[
        (
            estacoes_metrica[
                "cobertura_precipitacao_pct"
            ] >= COBERTURA_MINIMA
        )
        &
        (
            estacoes_metrica[
                "precipitacao_anual_mm"
            ].notna()
        )
    ]
    .copy()
)

print(
    "Estações-ano válidas para precipitação:"
)

print(
    len(
        estacoes_precipitacao_validas
    )
)

display(
    estacoes_precipitacao_validas[
        "ano"
    ]
    .value_counts()
    .sort_index()
)

Estações-ano válidas para precipitação:
719


ano
2019    152
2020    126
2021     84
2022     85
2023    145
2024    127
Name: count, dtype: int64

In [75]:
resumo_rede_precipitacao = (
    estacoes_precipitacao_validas
    .groupby(
        "ano"
    )
    .agg(
        estacoes_validas=(
            "codigo_wmo",
            "nunique"
        ),

        municipios_com_estacao=(
            "codigo_ibge",
            "nunique"
        )
    )
    .reset_index()
)

display(
    resumo_rede_precipitacao
)

,ano,estacoes_validas,municipios_com_estacao
0,2019,152,146
1,2020,126,121
2,2021,84,79
3,2022,85,80
4,2023,145,138
5,2024,127,120


In [76]:
centroides_dentro = (
    municipios_pontos[
        "geometry"
    ]
    .within(
        malha_metrica[
            "geometry"
        ]
    )
)

print(
    "Centroides dentro do próprio município:"
)

print(
    centroides_dentro.sum()
)

print(
    "\nCentroides fora do próprio município:"
)

print(
    (~centroides_dentro).sum()
)

Centroides dentro do próprio município:
1646

Centroides fora do próprio município:
15


In [77]:
pontos_referencia = (
    malha_metrica[
        "geometry"
    ]
    .centroid
)

pontos_internos = (
    malha_metrica[
        "geometry"
    ]
    .representative_point()
)

mascara_centroide_fora = (
    ~pontos_referencia.within(
        malha_metrica[
            "geometry"
        ]
    )
)

pontos_referencia.loc[
    mascara_centroide_fora
] = (
    pontos_internos.loc[
        mascara_centroide_fora
    ]
)

municipios_pontos = (
    malha_metrica[
        [
            "CD_MUN",
            "NM_MUN",
            "SIGLA_UF",
            "NM_REGIA",
            "AREA_KM2"
        ]
    ]
    .copy()
)

municipios_pontos = gpd.GeoDataFrame(
    municipios_pontos,
    geometry=pontos_referencia,
    crs=CRS_METRICO
)

print(
    "Municípios preparados:"
)

print(
    len(
        municipios_pontos
    )
)

print(
    "\nPontos fora após correção:"
)

print(
    (
        ~municipios_pontos[
            "geometry"
        ]
        .within(
            malha_metrica[
                "geometry"
            ]
        )
    ).sum()
)

Municípios preparados:
1661

Pontos fora após correção:
0


## 5.7 — Diagnóstico de proximidade da rede INMET

Antes da aplicação de qualquer método de estimativa espacial, será avaliada a
distância entre cada município e as estações meteorológicas disponíveis.

Para cada município e ano serão identificadas as três estações INMET válidas
mais próximas.

As distâncias são calculadas em CRS métrico (`EPSG:5880`) e posteriormente
convertidas para quilômetros.

Nesta primeira análise será utilizada a precipitação, considerando apenas
estações-ano com pelo menos 80% de cobertura válida para essa variável.

O objetivo desta etapa não é ainda estimar valores climáticos, mas analisar a
densidade espacial da rede e definir critérios metodológicos para a posterior
interpolação.

In [78]:
def calcular_estacoes_proximas(
    municipios,
    estacoes,
    ano,
    coluna_valor,
    coluna_cobertura,
    k=3
):

    estacoes_ano = (
        estacoes[
            estacoes[
                "ano"
            ] == ano
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    if len(estacoes_ano) < k:

        raise ValueError(
            f"{ano}: existem apenas "
            f"{len(estacoes_ano)} estações válidas."
        )


    municipios_temp = (
        municipios
        .copy()
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Coordenadas dos municípios
    # --------------------------------------------------------

    coords_municipios = np.column_stack(
        [
            municipios_temp.geometry.x.to_numpy(),
            municipios_temp.geometry.y.to_numpy()
        ]
    )


    # --------------------------------------------------------
    # Coordenadas das estações
    # --------------------------------------------------------

    coords_estacoes = np.column_stack(
        [
            estacoes_ano.geometry.x.to_numpy(),
            estacoes_ano.geometry.y.to_numpy()
        ]
    )


    # --------------------------------------------------------
    # Matriz de distâncias em metros
    #
    # dimensão:
    # municípios × estações
    # --------------------------------------------------------

    diferencas = (
        coords_municipios[:, None, :]
        -
        coords_estacoes[None, :, :]
    )

    distancias = np.sqrt(
        (
            diferencas ** 2
        ).sum(
            axis=2
        )
    )


    # --------------------------------------------------------
    # Índices das k estações mais próximas
    # --------------------------------------------------------

    indices_proximos = np.argpartition(
        distancias,
        kth=k - 1,
        axis=1
    )[
        :,
        :k
    ]


    distancias_proximas = np.take_along_axis(
        distancias,
        indices_proximos,
        axis=1
    )


    # Ordenação da mais próxima para a mais distante

    ordem = np.argsort(
        distancias_proximas,
        axis=1
    )

    indices_proximos = np.take_along_axis(
        indices_proximos,
        ordem,
        axis=1
    )

    distancias_proximas = np.take_along_axis(
        distancias_proximas,
        ordem,
        axis=1
    )


    # --------------------------------------------------------
    # Resultado
    # --------------------------------------------------------

    resultado = pd.DataFrame(
        {
            "codigo_ibge":
                municipios_temp[
                    "CD_MUN"
                ]
                .astype(str),

            "municipio":
                municipios_temp[
                    "NM_MUN"
                ],

            "uf":
                municipios_temp[
                    "SIGLA_UF"
                ],

            "regiao":
                municipios_temp[
                    "NM_REGIA"
                ],

            "ano":
                ano
        }
    )


    # --------------------------------------------------------
    # Informações das estações 1, 2 e 3
    # --------------------------------------------------------

    for posicao in range(k):

        idx = (
            indices_proximos[
                :,
                posicao
            ]
        )

        numero = (
            posicao + 1
        )


        resultado[
            f"wmo_{numero}"
        ] = (
            estacoes_ano.iloc[
                idx
            ][
                "codigo_wmo"
            ]
            .to_numpy()
        )


        resultado[
            f"estacao_{numero}"
        ] = (
            estacoes_ano.iloc[
                idx
            ][
                "estacao"
            ]
            .to_numpy()
        )


        resultado[
            f"uf_estacao_{numero}"
        ] = (
            estacoes_ano.iloc[
                idx
            ][
                "uf"
            ]
            .to_numpy()
        )


        resultado[
            f"valor_{numero}"
        ] = (
            estacoes_ano.iloc[
                idx
            ][
                coluna_valor
            ]
            .to_numpy()
        )


        resultado[
            f"cobertura_{numero}_pct"
        ] = (
            estacoes_ano.iloc[
                idx
            ][
                coluna_cobertura
            ]
            .to_numpy()
        )


        resultado[
            f"distancia_{numero}_km"
        ] = (
            distancias_proximas[
                :,
                posicao
            ]
            / 1000
        )


    return resultado

In [79]:
diagnostico_2019 = (
    calcular_estacoes_proximas(
        municipios=municipios_pontos,
        estacoes=estacoes_precipitacao_validas,
        ano=2019,
        coluna_valor="precipitacao_anual_mm",
        coluna_cobertura="cobertura_precipitacao_pct",
        k=3
    )
)

print(
    diagnostico_2019.shape
)

display(
    diagnostico_2019.head()
)

(1661, 23)


,codigo_ibge,municipio,uf,regiao,ano,wmo_1,estacao_1,uf_estacao_1,valor_1,cobertura_1_pct,...,uf_estacao_2,valor_2,cobertura_2_pct,distancia_2_km,wmo_3,estacao_3,uf_estacao_3,valor_3,cobertura_3_pct,distancia_3_km
0,5219902,São Francisco de Goiás,GO,Centro-oeste,2019,A002,GOIANIA,GO,1212.8,99.98,...,GO,993.8,89.03,86.001352,A014,GOIAS,GO,958.4,87.48,94.946721
1,4316204,Rondinha,RS,Sul,2019,A856,PALMEIRA DAS MISSOES,RS,1598.6,99.99,...,RS,1770.4,100.00,61.056759,A839,PASSO FUNDO,RS,1437.8,99.84,65.019557
2,4317558,Santo Antônio do Palma,RS,Sul,2019,A894,SERAFINA CORREA,RS,1751.8,100.00,...,RS,1437.8,99.84,48.858618,A844,LAGOA VERMELHA,RS,1600.6,98.88,57.274133
3,4209508,Laurentino,SC,Sul,2019,A863,ITUPORANGA,SC,1274.6,100.00,...,SC,1435.4,98.84,56.586139,A870,RANCHO QUEIMADO,SC,1329.6,85.97,86.066456
4,4202107,Barra Velha,SC,Sul,2019,A868,ITAJAI,SC,1587.6,100.00,...,SC,1435.4,98.84,60.577568,A862,RIO NEGRINHO,SC,1237.0,86.15,96.733362


In [80]:
display(
    diagnostico_2019[
        [
            "distancia_1_km",
            "distancia_2_km",
            "distancia_3_km"
        ]
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

,distancia_1_km,distancia_2_km,distancia_3_km
count,1661.000000,1661.000000,1661.000000
mean,43.913753,75.272174,96.823102
std,33.633401,43.405765,50.441941
min,0.974520,10.827084,24.829939
50%,36.827625,64.861082,83.906748
75%,53.664511,86.515766,109.039292
90%,76.364343,116.083833,145.487453
95%,95.050443,151.816756,199.929935
99%,179.469925,257.899561,307.272996
max,424.266273,513.968548,526.631462


In [81]:
display(
    diagnostico_2019[
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "wmo_1",
            "distancia_1_km",
            "wmo_2",
            "distancia_2_km",
            "wmo_3",
            "distancia_3_km"
        ]
    ]
    .sort_values(
        "distancia_3_km",
        ascending=False
    )
    .head(30)
)

,codigo_ibge,municipio,uf,wmo_1,distancia_1_km,wmo_2,distancia_2_km,wmo_3,distancia_3_km
1602,5107578,Rondolândia,MT,A927,379.821926,A928,513.968548,A922,526.631462
47,5103254,Colniza,MT,A927,424.266273,A924,461.071271,A926,497.174431
1637,5101407,Aripuanã,MT,A927,314.857928,A924,403.220591,A928,437.650482
1144,5107776,Santa Terezinha,MT,A942,173.879060,A005,377.193104,A930,414.592536
301,5108600,Vila Rica,MT,A942,119.607898,A906,382.843598,A930,408.005563
1201,5105150,Juína,MT,A927,162.038283,A928,294.830276,A924,379.911551
1448,5102694,Canabrava do Norte,MT,A942,99.527579,A930,269.276178,A906,363.700887
970,5106778,Porto Alegre do Norte,MT,A942,81.432178,A930,310.453043,A906,361.835330
1030,5103353,Confresa,MT,A942,74.314595,A930,351.486488,A906,354.105922
1127,5107743,Santa Cruz do Xingu,MT,A942,49.988248,A906,258.641023,A930,353.955102


In [82]:
diagnosticos_precipitacao = []

for ano in sorted(
    estacoes_precipitacao_validas[
        "ano"
    ].unique()
):

    print(
        f"Calculando distâncias - {ano}..."
    )

    diagnostico_ano = (
        calcular_estacoes_proximas(
            municipios=municipios_pontos,
            estacoes=estacoes_precipitacao_validas,
            ano=int(ano),
            coluna_valor="precipitacao_anual_mm",
            coluna_cobertura="cobertura_precipitacao_pct",
            k=3
        )
    )

    diagnosticos_precipitacao.append(
        diagnostico_ano
    )


diagnostico_precipitacao = pd.concat(
    diagnosticos_precipitacao,
    ignore_index=True
)

print(
    "\nDimensão final:"
)

print(
    diagnostico_precipitacao.shape
)

Calculando distâncias - 2019...
Calculando distâncias - 2020...
Calculando distâncias - 2021...
Calculando distâncias - 2022...
Calculando distâncias - 2023...
Calculando distâncias - 2024...

Dimensão final:
(9966, 23)


In [83]:
print(
    "Municípios:"
)

print(
    diagnostico_precipitacao[
        "codigo_ibge"
    ].nunique()
)

print(
    "\nAnos:"
)

print(
    sorted(
        diagnostico_precipitacao[
            "ano"
        ].unique()
    )
)

print(
    "\nDuplicidades município + ano:"
)

print(
    diagnostico_precipitacao[
        [
            "codigo_ibge",
            "ano"
        ]
    ]
    .duplicated()
    .sum()
)

Municípios:
1661

Anos:
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Duplicidades município + ano:
0


In [84]:
resumo_distancias = []

for ano, grupo in (
    diagnostico_precipitacao
    .groupby(
        "ano"
    )
):

    for coluna in [
        "distancia_1_km",
        "distancia_2_km",
        "distancia_3_km"
    ]:

        serie = (
            grupo[
                coluna
            ]
            .dropna()
        )

        resumo_distancias.append(
            {
                "ano":
                    ano,

                "distancia":
                    coluna,

                "media_km":
                    serie.mean(),

                "mediana_km":
                    serie.median(),

                "p75_km":
                    serie.quantile(
                        0.75
                    ),

                "p90_km":
                    serie.quantile(
                        0.90
                    ),

                "p95_km":
                    serie.quantile(
                        0.95
                    ),

                "p99_km":
                    serie.quantile(
                        0.99
                    ),

                "max_km":
                    serie.max()
            }
        )


resumo_distancias = pd.DataFrame(
    resumo_distancias
)

display(
    resumo_distancias
)

,ano,distancia,media_km,mediana_km,p75_km,p90_km,p95_km,p99_km,max_km
0,2019,distancia_1_km,43.913753,36.827625,53.664511,76.364343,95.050443,179.469925,424.266273
1,2019,distancia_2_km,75.272174,64.861082,86.515766,116.083833,151.816756,257.899561,513.968548
2,2019,distancia_3_km,96.823102,83.906748,109.039292,145.487453,199.929935,307.272996,526.631462
3,2020,distancia_1_km,49.240388,40.903752,60.538079,89.955891,114.421119,194.702260,285.677370
4,2020,distancia_2_km,83.114521,69.719681,97.579479,133.733296,181.054142,284.570312,476.390677
5,2020,distancia_3_km,107.699251,91.822751,127.246460,168.509528,217.060146,339.412590,494.544354
6,2021,distancia_1_km,77.620469,52.329134,94.403987,160.139404,213.503820,451.349120,697.106092
7,2021,distancia_2_km,123.904883,86.192402,165.068865,239.600276,297.190408,510.634273,700.797148
8,2021,distancia_3_km,154.683047,109.486792,200.523061,279.170917,420.754698,628.130710,834.113490
9,2022,distancia_1_km,71.635559,50.806918,84.013599,144.498015,192.331954,410.245109,565.664243


In [85]:
limites_km = [
    50,
    100,
    150,
    200,
    300
]

cobertura_por_distancia = []

for ano, grupo in (
    diagnostico_precipitacao
    .groupby(
        "ano"
    )
):

    for limite in limites_km:

        dentro = (
            grupo[
                "distancia_3_km"
            ] <= limite
        )

        cobertura_por_distancia.append(
            {
                "ano":
                    ano,

                "limite_km":
                    limite,

                "municipios":
                    int(
                        dentro.sum()
                    ),

                "total_municipios":
                    len(
                        grupo
                    ),

                "percentual":
                    round(
                        dentro.mean()
                        * 100,
                        2
                    )
            }
        )


cobertura_por_distancia = pd.DataFrame(
    cobertura_por_distancia
)

display(
    cobertura_por_distancia
)

,ano,limite_km,municipios,total_municipios,percentual
0,2019,50,76,1661,4.58
1,2019,100,1118,1661,67.31
2,2019,150,1508,1661,90.79
3,2019,200,1579,1661,95.06
4,2019,300,1642,1661,98.86
5,2020,50,65,1661,3.91
6,2020,100,942,1661,56.71
7,2020,150,1422,1661,85.61
8,2020,200,1554,1661,93.56
9,2020,300,1629,1661,98.07


In [86]:
ARQUIVO_DIAGNOSTICO_DISTANCIAS = (
    QUALITY_DIR
    / "diagnostico_distancias_precipitacao_2019_2024.csv"
)

diagnostico_precipitacao.to_csv(
    ARQUIVO_DIAGNOSTICO_DISTANCIAS,
    index=False,
    encoding="utf-8-sig"
)


resumo_distancias.to_csv(
    QUALITY_DIR
    / "resumo_distancias_precipitacao_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)


cobertura_por_distancia.to_csv(
    QUALITY_DIR
    / "cobertura_3_estacoes_por_raio_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Diagnóstico espacial salvo."
)

Diagnóstico espacial salvo.


In [87]:
limites_km = [
    100,
    150,
    200,
    300
]

colunas_distancia = {
    1: "distancia_1_km",
    2: "distancia_2_km",
    3: "distancia_3_km"
}

cobertura_k_raio = []

for ano, grupo in (
    diagnostico_precipitacao
    .groupby("ano")
):

    for k, coluna in (
        colunas_distancia.items()
    ):

        for limite in limites_km:

            mascara = (
                grupo[coluna]
                <= limite
            )

            cobertura_k_raio.append(
                {
                    "ano": int(ano),
                    "k_estacoes": k,
                    "limite_km": limite,

                    "municipios":
                        int(
                            mascara.sum()
                        ),

                    "total_municipios":
                        len(grupo),

                    "percentual":
                        round(
                            mascara.mean()
                            * 100,
                            2
                        )
                }
            )


cobertura_k_raio = pd.DataFrame(
    cobertura_k_raio
)

display(
    cobertura_k_raio
)

,ano,k_estacoes,limite_km,municipios,total_municipios,percentual
0,2019,1,100,1587,1661,95.54
1,2019,1,150,1633,1661,98.31
2,2019,1,200,1649,1661,99.28
3,2019,1,300,1658,1661,99.82
4,2019,2,100,1390,1661,83.68
...,...,...,...,...,...,...
67,2024,2,300,1657,1661,99.76
68,2024,3,100,954,1661,57.44
69,2024,3,150,1368,1661,82.36
70,2024,3,200,1524,1661,91.75


In [88]:
tabela_cobertura_200 = (
    cobertura_k_raio[
        cobertura_k_raio[
            "limite_km"
        ] == 200
    ]
    .pivot(
        index="ano",
        columns="k_estacoes",
        values="percentual"
    )
    .rename(
        columns={
            1: "1_estacao_pct",
            2: "2_estacoes_pct",
            3: "3_estacoes_pct"
        }
    )
)

display(
    tabela_cobertura_200
)

k_estacoes,1_estacao_pct,2_estacoes_pct,3_estacoes_pct
ano,,,
2019,99.28,98.13,95.06
2020,99.10,96.27,93.56
2021,94.16,83.20,74.83
2022,95.61,88.68,78.57
2023,99.10,97.59,94.04
2024,98.92,96.39,91.75


In [89]:
tabela_cobertura_300 = (
    cobertura_k_raio[
        cobertura_k_raio[
            "limite_km"
        ] == 300
    ]
    .pivot(
        index="ano",
        columns="k_estacoes",
        values="percentual"
    )
    .rename(
        columns={
            1: "1_estacao_pct",
            2: "2_estacoes_pct",
            3: "3_estacoes_pct"
        }
    )
)

display(
    tabela_cobertura_300
)

k_estacoes,1_estacao_pct,2_estacoes_pct,3_estacoes_pct
ano,,,
2019,99.82,99.46,98.86
2020,100.00,99.16,98.07
2021,97.29,95.00,91.81
2022,97.83,96.99,95.00
2023,99.82,99.76,99.34
2024,99.94,99.76,98.62


In [90]:
cobertura_k_raio.to_csv(
    QUALITY_DIR
    / "cobertura_rede_por_k_e_raio_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Diagnóstico por quantidade de estações salvo."
)

Diagnóstico por quantidade de estações salvo.


In [92]:
POTENCIA_IDW = 2
RAIO_PREFERENCIAL_KM = 200
RAIO_EXPANDIDO_KM = 300

In [93]:
def calcular_idw(
    valores,
    distancias_km,
    potencia=2
):

    valores = np.asarray(
        valores,
        dtype=float
    )

    distancias = np.asarray(
        distancias_km,
        dtype=float
    )


    mascara = (
        ~np.isnan(valores)
        &
        ~np.isnan(distancias)
    )

    valores = valores[
        mascara
    ]

    distancias = distancias[
        mascara
    ]


    if len(valores) == 0:
        return np.nan


    # proteção caso exista distância zero
    if np.any(
        distancias == 0
    ):

        return valores[
            np.argmin(
                distancias
            )
        ]


    pesos = (
        1
        /
        np.power(
            distancias,
            potencia
        )
    )


    return np.sum(
        pesos
        * valores
    ) / np.sum(
        pesos
    )

In [94]:
def estimar_precipitacao_linha(
    linha,
    potencia=2,
    raio_preferencial=200,
    raio_expandido=300
):

    candidatos = []

    for i in [
        1,
        2,
        3
    ]:

        valor = linha[
            f"valor_{i}"
        ]

        distancia = linha[
            f"distancia_{i}_km"
        ]

        wmo = linha[
            f"wmo_{i}"
        ]

        if (
            pd.notna(valor)
            and
            pd.notna(distancia)
        ):

            candidatos.append(
                {
                    "wmo": wmo,
                    "valor": valor,
                    "distancia": distancia
                }
            )


    # ====================================================
    # ETAPA 1
    # estações até 200 km
    # ====================================================

    dentro_200 = [
        item
        for item in candidatos
        if item[
            "distancia"
        ] <= raio_preferencial
    ]


    if len(dentro_200) >= 2:

        usados = (
            dentro_200[
                :3
            ]
        )

        valor_estimado = calcular_idw(
            [
                x["valor"]
                for x in usados
            ],
            [
                x["distancia"]
                for x in usados
            ],
            potencia=potencia
        )

        return pd.Series(
            {
                "precipitacao_estimada_mm":
                    valor_estimado,

                "numero_estacoes_estimativa":
                    len(usados),

                "distancia_max_estacoes_km":
                    max(
                        x["distancia"]
                        for x in usados
                    ),

                "qualidade_estimativa":
                    (
                        "alta"
                        if len(usados) == 3
                        else "media"
                    )
            }
        )


    # ====================================================
    # ETAPA 2
    # amplia até 300 km
    # ====================================================

    dentro_300 = [
        item
        for item in candidatos
        if item[
            "distancia"
        ] <= raio_expandido
    ]


    if len(dentro_300) >= 2:

        usados = (
            dentro_300[
                :3
            ]
        )

        valor_estimado = calcular_idw(
            [
                x["valor"]
                for x in usados
            ],
            [
                x["distancia"]
                for x in usados
            ],
            potencia=potencia
        )

        return pd.Series(
            {
                "precipitacao_estimada_mm":
                    valor_estimado,

                "numero_estacoes_estimativa":
                    len(usados),

                "distancia_max_estacoes_km":
                    max(
                        x["distancia"]
                        for x in usados
                    ),

                "qualidade_estimativa":
                    "baixa"
            }
        )


    # ====================================================
    # ETAPA 3
    # somente uma estação até 300 km
    # ====================================================

    if len(dentro_300) == 1:

        unico = (
            dentro_300[0]
        )

        return pd.Series(
            {
                "precipitacao_estimada_mm":
                    unico[
                        "valor"
                    ],

                "numero_estacoes_estimativa":
                    1,

                "distancia_max_estacoes_km":
                    unico[
                        "distancia"
                    ],

                "qualidade_estimativa":
                    "muito_baixa"
            }
        )


    # ====================================================
    # ETAPA 4
    # nenhuma até 300 km
    #
    # fallback:
    # estação válida mais próxima
    # ====================================================

    if len(candidatos) > 0:

        mais_proxima = min(
            candidatos,
            key=lambda x: x[
                "distancia"
            ]
        )

        return pd.Series(
            {
                "precipitacao_estimada_mm":
                    mais_proxima[
                        "valor"
                    ],

                "numero_estacoes_estimativa":
                    1,

                "distancia_max_estacoes_km":
                    mais_proxima[
                        "distancia"
                    ],

                "qualidade_estimativa":
                    "muito_baixa"
            }
        )


    return pd.Series(
        {
            "precipitacao_estimada_mm":
                np.nan,

            "numero_estacoes_estimativa":
                0,

            "distancia_max_estacoes_km":
                np.nan,

            "qualidade_estimativa":
                "sem_estimativa"
        }
    )

In [95]:
resultado_idw_precipitacao = (
    diagnostico_precipitacao
    .apply(
        estimar_precipitacao_linha,
        axis=1
    )
)

In [96]:
precipitacao_estimativa = (
    pd.concat(
        [
            diagnostico_precipitacao[
                [
                    "codigo_ibge",
                    "municipio",
                    "uf",
                    "regiao",
                    "ano"
                ]
            ]
            .reset_index(
                drop=True
            ),

            resultado_idw_precipitacao
            .reset_index(
                drop=True
            )
        ],
        axis=1
    )
)

print(
    precipitacao_estimativa.shape
)

display(
    precipitacao_estimativa.head()
)

(9966, 9)


,codigo_ibge,municipio,uf,regiao,ano,precipitacao_estimada_mm,numero_estacoes_estimativa,distancia_max_estacoes_km,qualidade_estimativa
0,5219902,São Francisco de Goiás,GO,Centro-oeste,2019,1073.612878,3,94.946721,alta
1,4316204,Rondinha,RS,Sul,2019,1606.290803,3,65.019557,alta
2,4317558,Santo Antônio do Palma,RS,Sul,2019,1667.721094,3,57.274133,alta
3,4209508,Laurentino,SC,Sul,2019,1302.575419,3,86.066456,alta
4,4202107,Barra Velha,SC,Sul,2019,1528.704529,3,96.733362,alta


In [97]:
display(
    precipitacao_estimativa[
        "qualidade_estimativa"
    ]
    .value_counts()
)

qualidade_estimativa
alta           8767
media           539
baixa           496
muito_baixa     164
Name: count, dtype: int64

In [98]:
display(
    precipitacao_estimativa[
        "qualidade_estimativa"
    ]
    .value_counts(
        normalize=True
    )
    .mul(100)
    .round(2)
    .rename(
        "percentual"
    )
)

qualidade_estimativa
alta           87.97
media           5.41
baixa           4.98
muito_baixa     1.65
Name: percentual, dtype: float64

In [99]:
qualidade_por_ano = (
    precipitacao_estimativa
    .groupby(
        [
            "ano",
            "qualidade_estimativa"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

display(
    qualidade_por_ano
)

qualidade_estimativa,alta,baixa,media,muito_baixa
ano,,,,
2019,1579,22,51,9
2020,1554,48,45,14
2021,1243,196,139,83
2022,1305,138,168,50
2023,1562,36,59,4
2024,1524,56,77,4


In [100]:
display(
    precipitacao_estimativa[
        precipitacao_estimativa[
            "qualidade_estimativa"
        ] == "muito_baixa"
    ]
    .sort_values(
        "distancia_max_estacoes_km",
        ascending=False
    )
    .head(30)
)

,codigo_ibge,municipio,uf,regiao,ano,precipitacao_estimada_mm,numero_estacoes_estimativa,distancia_max_estacoes_km,qualidade_estimativa
4449,5107743,Santa Cruz do Xingu,MT,Centro-oeste,2021,1433.0,1,697.106092,muito_baixa
3664,5107354,São José do Xingu,MT,Centro-oeste,2021,1788.0,1,639.600556,muito_baixa
4352,5103353,Confresa,MT,Centro-oeste,2021,1488.0,1,611.973898,muito_baixa
3623,5108600,Vila Rica,MT,Centro-oeste,2021,1488.0,1,611.958719,muito_baixa
4292,5106778,Porto Alegre do Norte,MT,Centro-oeste,2021,1836.0,1,584.842040,muito_baixa
4473,5106422,Peixoto de Azevedo,MT,Centro-oeste,2021,1433.0,1,582.683257,muito_baixa
5852,5104104,Guarantã do Norte,MT,Centro-oeste,2022,792.0,1,565.664243,muito_baixa
4659,5107859,São Félix do Araguaia,MT,Centro-oeste,2021,1788.0,1,555.761045,muito_baixa
4770,5102694,Canabrava do Norte,MT,Centro-oeste,2021,1788.0,1,555.574617,muito_baixa
5452,5105606,Matupá,MT,Centro-oeste,2022,1679.6,1,537.567841,muito_baixa


In [101]:
precipitacao_observada = (
    inmet_municipio_ano_observado[
        [
            "codigo_ibge",
            "ano",
            "precipitacao_anual_mm",
            "numero_estacoes_precipitacao_validas",
            "cobertura_precipitacao_media_pct"
        ]
    ]
    .copy()
)

In [102]:
precipitacao_final = (
    precipitacao_estimativa
    .merge(
        precipitacao_observada,
        on=[
            "codigo_ibge",
            "ano"
        ],
        how="left"
    )
)

In [103]:
precipitacao_final[
    "origem_precipitacao"
] = np.where(
    precipitacao_final[
        "precipitacao_anual_mm"
    ].notna(),

    "observado_inmet",

    "estimado_idw"
)

In [104]:
precipitacao_final[
    "qualidade_precipitacao"
] = (
    precipitacao_final[
        "qualidade_estimativa"
    ]
)

mascara_observado = (
    precipitacao_final[
        "origem_precipitacao"
    ] == "observado_inmet"
)

precipitacao_final.loc[
    mascara_observado,
    "qualidade_precipitacao"
] = (
    "observado"
)

In [105]:
display(
    precipitacao_final[
        "origem_precipitacao"
    ]
    .value_counts()
)

origem_precipitacao
estimado_idw       9282
observado_inmet     684
Name: count, dtype: int64

In [106]:
display(
    precipitacao_final[
        "qualidade_precipitacao"
    ]
    .value_counts()
)

qualidade_precipitacao
alta           8133
observado       684
media           501
baixa           486
muito_baixa     162
Name: count, dtype: int64

In [110]:
# ============================================================
# CRIAÇÃO DO VALOR FINAL DE PRECIPITAÇÃO
#
# Prioridade:
# 1. valor observado pelo INMET quando disponível
# 2. valor estimado por IDW quando não houver observação válida
# ============================================================

precipitacao_final[
    "precipitacao_final_mm"
] = (
    precipitacao_final[
        "precipitacao_anual_mm"
    ]
    .combine_first(
        precipitacao_final[
            "precipitacao_estimada_mm"
        ]
    )
)

print(
    "Coluna precipitacao_final_mm criada."
)

display(
    precipitacao_final[
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "ano",
            "precipitacao_anual_mm",
            "precipitacao_estimada_mm",
            "precipitacao_final_mm",
            "origem_precipitacao",
            "qualidade_precipitacao"
        ]
    ].head(10)
)

Coluna precipitacao_final_mm criada.


,codigo_ibge,municipio,uf,ano,precipitacao_anual_mm,precipitacao_estimada_mm,precipitacao_final_mm,origem_precipitacao,qualidade_precipitacao
0,5219902,São Francisco de Goiás,GO,2019,NaN,1073.612878,1073.612878,estimado_idw,alta
1,4316204,Rondinha,RS,2019,NaN,1606.290803,1606.290803,estimado_idw,alta
2,4317558,Santo Antônio do Palma,RS,2019,NaN,1667.721094,1667.721094,estimado_idw,alta
3,4209508,Laurentino,SC,2019,NaN,1302.575419,1302.575419,estimado_idw,alta
4,4202107,Barra Velha,SC,2019,NaN,1528.704529,1528.704529,estimado_idw,alta
5,4217808,Taió,SC,2019,NaN,1412.927905,1412.927905,estimado_idw,alta
6,4319307,São Paulo das Missões,RS,2019,NaN,1870.235128,1870.235128,estimado_idw,alta
7,5215652,Palestina de Goiás,GO,2019,NaN,1248.582639,1248.582639,estimado_idw,alta
8,4311429,Lajeado do Bugre,RS,2019,NaN,1595.735634,1595.735634,estimado_idw,alta
9,4318002,São Borja,RS,2019,1632.8,1681.947688,1632.800000,observado_inmet,observado


In [111]:
print(
    precipitacao_final.columns.tolist()
)

['codigo_ibge', 'municipio', 'uf', 'regiao', 'ano', 'precipitacao_estimada_mm', 'numero_estacoes_estimativa', 'distancia_max_estacoes_km', 'qualidade_estimativa', 'precipitacao_anual_mm', 'numero_estacoes_precipitacao_validas', 'cobertura_precipitacao_media_pct', 'origem_precipitacao', 'qualidade_precipitacao', 'precipitacao_final_mm']


In [112]:
print(
    "Dimensão:"
)

print(
    precipitacao_final.shape
)


print(
    "\nMunicípios:"
)

print(
    precipitacao_final[
        "codigo_ibge"
    ].nunique()
)


print(
    "\nAnos:"
)

print(
    precipitacao_final[
        "ano"
    ].nunique()
)


print(
    "\nDuplicidades:"
)

print(
    precipitacao_final[
        [
            "codigo_ibge",
            "ano"
        ]
    ]
    .duplicated()
    .sum()
)


print(
    "\nPrecipitação final ausente:"
)

print(
    precipitacao_final[
        "precipitacao_final_mm"
    ]
    .isna()
    .sum()
)

Dimensão:
(9966, 15)

Municípios:
1661

Anos:
6

Duplicidades:
0

Precipitação final ausente:
0


In [113]:
teste_prioridade_observado = (
    precipitacao_final[
        precipitacao_final[
            "origem_precipitacao"
        ] == "observado_inmet"
    ]
)

print(
    "Registros observados:"
)

print(
    len(
        teste_prioridade_observado
    )
)


print(
    "\nDiferenças entre observado e valor final:"
)

print(
    (
        teste_prioridade_observado[
            "precipitacao_anual_mm"
        ]
        !=
        teste_prioridade_observado[
            "precipitacao_final_mm"
        ]
    ).sum()
)

Registros observados:
684

Diferenças entre observado e valor final:
0


In [114]:
teste_estimados = (
    precipitacao_final[
        precipitacao_final[
            "origem_precipitacao"
        ] == "estimado_idw"
    ]
)

print(
    "Registros estimados:"
)

print(
    len(
        teste_estimados
    )
)


print(
    "\nEstimativas ausentes:"
)

print(
    teste_estimados[
        "precipitacao_final_mm"
    ]
    .isna()
    .sum()
)

Registros estimados:
9282

Estimativas ausentes:
0


In [115]:
resumo_origem_precipitacao = (
    precipitacao_final[
        "origem_precipitacao"
    ]
    .value_counts()
    .rename_axis(
        "origem"
    )
    .reset_index(
        name="registros"
    )
)

resumo_origem_precipitacao[
    "percentual"
] = (
    resumo_origem_precipitacao[
        "registros"
    ]
    /
    len(
        precipitacao_final
    )
    * 100
)

resumo_origem_precipitacao[
    "percentual"
] = (
    resumo_origem_precipitacao[
        "percentual"
    ]
    .round(2)
)

display(
    resumo_origem_precipitacao
)

,origem,registros,percentual
0,estimado_idw,9282,93.14
1,observado_inmet,684,6.86


In [116]:
resumo_qualidade_precipitacao = (
    precipitacao_final[
        "qualidade_precipitacao"
    ]
    .value_counts()
    .rename_axis(
        "qualidade"
    )
    .reset_index(
        name="registros"
    )
)

resumo_qualidade_precipitacao[
    "percentual"
] = (
    resumo_qualidade_precipitacao[
        "registros"
    ]
    /
    len(
        precipitacao_final
    )
    * 100
)

resumo_qualidade_precipitacao[
    "percentual"
] = (
    resumo_qualidade_precipitacao[
        "percentual"
    ]
    .round(2)
)

display(
    resumo_qualidade_precipitacao
)

,qualidade,registros,percentual
0,alta,8133,81.61
1,observado,684,6.86
2,media,501,5.03
3,baixa,486,4.88
4,muito_baixa,162,1.63


In [117]:
display(
    precipitacao_final[
        "precipitacao_final_mm"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

count    9966.000000
mean     1542.578403
std       427.858376
min       348.400000
1%        769.597445
5%        968.830875
25%      1233.537070
50%      1464.514595
75%      1805.623592
95%      2352.477548
99%      2659.864800
max      3394.200000
Name: precipitacao_final_mm, dtype: float64

In [118]:
print(
    "Menores precipitações:"
)

display(
    precipitacao_final[
        [
            "municipio",
            "uf",
            "ano",
            "precipitacao_final_mm",
            "origem_precipitacao",
            "qualidade_precipitacao"
        ]
    ]
    .sort_values(
        "precipitacao_final_mm"
    )
    .head(20)
)


print(
    "\nMaiores precipitações:"
)

display(
    precipitacao_final[
        [
            "municipio",
            "uf",
            "ano",
            "precipitacao_final_mm",
            "origem_precipitacao",
            "qualidade_precipitacao"
        ]
    ]
    .sort_values(
        "precipitacao_final_mm",
        ascending=False
    )
    .head(20)
)

Menores precipitações:


,municipio,uf,ano,precipitacao_final_mm,origem_precipitacao,qualidade_precipitacao
6821,Monte Alegre de Goiás,GO,2023,348.400000,observado_inmet,observado
7301,Divinópolis de Goiás,GO,2023,401.647990,estimado_idw,media
6697,Campos Belos,GO,2023,409.790444,estimado_idw,media
1964,Sidrolândia,MS,2020,424.400000,observado_inmet,observado
7227,São Domingos,GO,2023,457.624620,estimado_idw,media
2210,Rio Brilhante,MS,2020,464.400000,observado_inmet,observado
8820,Alto Paraíso de Goiás,GO,2024,468.200000,observado_inmet,observado
2748,São Mateus do Sul,PR,2020,518.000000,observado_inmet,observado
3871,Rio Brilhante,MS,2021,535.600000,observado_inmet,observado
5259,Guiratinga,MT,2022,581.600000,observado_inmet,observado



Maiores precipitações:


,municipio,uf,ano,precipitacao_final_mm,origem_precipitacao,qualidade_precipitacao
7407,Bom Jardim da Serra,SC,2023,3394.200000,observado_inmet,observado
8929,Canela,RS,2024,3310.200000,observado_inmet,observado
8410,Gramado,RS,2024,3256.882055,estimado_idw,alta
9068,Bom Jardim da Serra,SC,2024,3218.400000,observado_inmet,observado
7056,Urubici,SC,2023,3216.896129,estimado_idw,alta
7606,Grão-Pará,SC,2023,3159.048543,estimado_idw,alta
9610,Três Coroas,RS,2024,3124.597717,estimado_idw,alta
8717,Urubici,SC,2024,3081.009972,estimado_idw,alta
9267,Grão-Pará,SC,2024,3015.091533,estimado_idw,alta
7768,Rio Fortuna,SC,2023,2923.441981,estimado_idw,alta


In [119]:
ARQUIVO_PRECIPITACAO_CURATED = (
    CURATED_MUNICIPIO_ANO_DIR
    / "inmet_precipitacao_municipio_ano_2019_2024.csv"
)

precipitacao_final.to_csv(
    ARQUIVO_PRECIPITACAO_CURATED,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Camada municipal de precipitação salva:"
)

print(
    ARQUIVO_PRECIPITACAO_CURATED
)

Camada municipal de precipitação salva:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\inmet\municipio_ano\inmet_precipitacao_municipio_ano_2019_2024.csv


In [120]:
precipitacao_final[
    "qualidade_espacial_precipitacao"
] = (
    precipitacao_final[
        "qualidade_precipitacao"
    ]
)

## 5.8 — Generalização da estimativa espacial climática

A metodologia espacial validada inicialmente para precipitação será reutilizada
para outras variáveis climáticas.

Cada variável será tratada separadamente porque a disponibilidade dos sensores
e a cobertura temporal variam entre precipitação, temperatura e umidade.

Critério de qualidade adotado no projeto:

- cobertura válida mínima: 80%;
- até 3 estações mais próximas;
- raio preferencial: 200 km;
- raio expandido: 300 km;
- interpolação por distância inversa (IDW);
- potência IDW: 2.

Valores observados diretamente em estações localizadas no município possuem
prioridade sobre valores estimados espacialmente.

As classes `alta`, `media`, `baixa` e `muito_baixa` representam qualidade
espacial da estimativa e não certificação da precisão meteorológica.

In [121]:
def estimar_variavel_linha(
    linha,
    nome_saida,
    potencia=2,
    raio_preferencial=200,
    raio_expandido=300
):

    candidatos = []

    for i in [1, 2, 3]:

        valor = linha[
            f"valor_{i}"
        ]

        distancia = linha[
            f"distancia_{i}_km"
        ]

        wmo = linha[
            f"wmo_{i}"
        ]

        if (
            pd.notna(valor)
            and
            pd.notna(distancia)
        ):

            candidatos.append(
                {
                    "wmo": wmo,
                    "valor": valor,
                    "distancia": distancia
                }
            )


    # ====================================================
    # Até 200 km
    # ====================================================

    dentro_200 = [
        item
        for item in candidatos
        if item["distancia"]
        <= raio_preferencial
    ]

    if len(dentro_200) >= 2:

        usados = dentro_200[:3]

        valor_estimado = calcular_idw(
            [
                x["valor"]
                for x in usados
            ],
            [
                x["distancia"]
                for x in usados
            ],
            potencia=potencia
        )

        qualidade = (
            "alta"
            if len(usados) == 3
            else "media"
        )

        return pd.Series(
            {
                nome_saida:
                    valor_estimado,

                "numero_estacoes_estimativa":
                    len(usados),

                "distancia_max_estacoes_km":
                    max(
                        x["distancia"]
                        for x in usados
                    ),

                "qualidade_espacial_estimativa":
                    qualidade
            }
        )


    # ====================================================
    # Amplia até 300 km
    # ====================================================

    dentro_300 = [
        item
        for item in candidatos
        if item["distancia"]
        <= raio_expandido
    ]

    if len(dentro_300) >= 2:

        usados = dentro_300[:3]

        valor_estimado = calcular_idw(
            [
                x["valor"]
                for x in usados
            ],
            [
                x["distancia"]
                for x in usados
            ],
            potencia=potencia
        )

        return pd.Series(
            {
                nome_saida:
                    valor_estimado,

                "numero_estacoes_estimativa":
                    len(usados),

                "distancia_max_estacoes_km":
                    max(
                        x["distancia"]
                        for x in usados
                    ),

                "qualidade_espacial_estimativa":
                    "baixa"
            }
        )


    # ====================================================
    # Somente uma estação até 300 km
    # ====================================================

    if len(dentro_300) == 1:

        unico = dentro_300[0]

        return pd.Series(
            {
                nome_saida:
                    unico["valor"],

                "numero_estacoes_estimativa":
                    1,

                "distancia_max_estacoes_km":
                    unico["distancia"],

                "qualidade_espacial_estimativa":
                    "muito_baixa"
            }
        )


    # ====================================================
    # Nenhuma estação até 300 km
    # fallback para a mais próxima
    # ====================================================

    if len(candidatos) > 0:

        mais_proxima = min(
            candidatos,
            key=lambda x: x["distancia"]
        )

        return pd.Series(
            {
                nome_saida:
                    mais_proxima["valor"],

                "numero_estacoes_estimativa":
                    1,

                "distancia_max_estacoes_km":
                    mais_proxima["distancia"],

                "qualidade_espacial_estimativa":
                    "muito_baixa"
            }
        )


    return pd.Series(
        {
            nome_saida:
                np.nan,

            "numero_estacoes_estimativa":
                0,

            "distancia_max_estacoes_km":
                np.nan,

            "qualidade_espacial_estimativa":
                "sem_estimativa"
        }
    )

In [122]:
estacoes_temperatura_validas = (
    estacoes_metrica[
        (
            estacoes_metrica[
                "cobertura_temperatura_pct"
            ] >= COBERTURA_MINIMA
        )
        &
        (
            estacoes_metrica[
                "temperatura_media_anual_c"
            ].notna()
        )
    ]
    .copy()
)

print(
    "Estações-ano válidas para temperatura:"
)

print(
    len(
        estacoes_temperatura_validas
    )
)

display(
    estacoes_temperatura_validas[
        "ano"
    ]
    .value_counts()
    .sort_index()
)

Estações-ano válidas para temperatura:
810


ano
2019    164
2020    141
2021    101
2022    100
2023    160
2024    144
Name: count, dtype: int64

In [123]:
resumo_rede_temperatura = (
    estacoes_temperatura_validas
    .groupby(
        "ano"
    )
    .agg(
        estacoes_validas=(
            "codigo_wmo",
            "nunique"
        ),

        municipios_com_estacao=(
            "codigo_ibge",
            "nunique"
        )
    )
    .reset_index()
)

display(
    resumo_rede_temperatura
)

,ano,estacoes_validas,municipios_com_estacao
0,2019,164,158
1,2020,141,136
2,2021,101,96
3,2022,100,95
4,2023,160,153
5,2024,144,137


In [124]:
diagnosticos_temperatura = []

for ano in sorted(
    estacoes_temperatura_validas[
        "ano"
    ].unique()
):

    print(
        f"Calculando temperatura - {ano}..."
    )

    diagnostico_ano = (
        calcular_estacoes_proximas(
            municipios=municipios_pontos,
            estacoes=estacoes_temperatura_validas,
            ano=int(ano),
            coluna_valor="temperatura_media_anual_c",
            coluna_cobertura="cobertura_temperatura_pct",
            k=3
        )
    )

    diagnosticos_temperatura.append(
        diagnostico_ano
    )


diagnostico_temperatura = pd.concat(
    diagnosticos_temperatura,
    ignore_index=True
)

print(
    "\nDimensão:"
)

print(
    diagnostico_temperatura.shape
)

Calculando temperatura - 2019...
Calculando temperatura - 2020...
Calculando temperatura - 2021...
Calculando temperatura - 2022...
Calculando temperatura - 2023...
Calculando temperatura - 2024...

Dimensão:
(9966, 23)


In [125]:
resultado_idw_temperatura = (
    diagnostico_temperatura
    .apply(
        lambda linha:
            estimar_variavel_linha(
                linha,
                nome_saida=
                    "temperatura_estimada_c",
                potencia=
                    POTENCIA_IDW,
                raio_preferencial=
                    RAIO_PREFERENCIAL_KM,
                raio_expandido=
                    RAIO_EXPANDIDO_KM
            ),
        axis=1
    )
)

In [126]:
temperatura_estimativa = pd.concat(
    [
        diagnostico_temperatura[
            [
                "codigo_ibge",
                "municipio",
                "uf",
                "regiao",
                "ano"
            ]
        ]
        .reset_index(
            drop=True
        ),

        resultado_idw_temperatura
        .reset_index(
            drop=True
        )
    ],
    axis=1
)

print(
    temperatura_estimativa.shape
)

display(
    temperatura_estimativa.head()
)

(9966, 9)


,codigo_ibge,municipio,uf,regiao,ano,temperatura_estimada_c,numero_estacoes_estimativa,distancia_max_estacoes_km,qualidade_espacial_estimativa
0,5219902,São Francisco de Goiás,GO,Centro-oeste,2019,25.089393,3,94.946721,alta
1,4316204,Rondinha,RS,Sul,2019,18.880873,3,65.019557,alta
2,4317558,Santo Antônio do Palma,RS,Sul,2019,18.313975,3,57.274133,alta
3,4209508,Laurentino,SC,Sul,2019,19.067521,3,86.066456,alta
4,4202107,Barra Velha,SC,Sul,2019,20.708063,3,96.733362,alta


In [127]:
temperatura_observada = (
    inmet_municipio_ano_observado[
        [
            "codigo_ibge",
            "ano",
            "temperatura_media_anual_c",
            "numero_estacoes_temperatura_validas",
            "cobertura_temperatura_media_pct"
        ]
    ]
    .copy()
)

In [128]:
temperatura_final = (
    temperatura_estimativa
    .merge(
        temperatura_observada,
        on=[
            "codigo_ibge",
            "ano"
        ],
        how="left"
    )
)

In [129]:
temperatura_final[
    "temperatura_final_c"
] = (
    temperatura_final[
        "temperatura_media_anual_c"
    ]
    .combine_first(
        temperatura_final[
            "temperatura_estimada_c"
        ]
    )
)

In [130]:
temperatura_final[
    "origem_temperatura"
] = np.where(
    temperatura_final[
        "temperatura_media_anual_c"
    ].notna(),

    "observado_inmet",

    "estimado_idw"
)

In [131]:
temperatura_final[
    "qualidade_temperatura"
] = (
    temperatura_final[
        "qualidade_espacial_estimativa"
    ]
)

mascara_observada = (
    temperatura_final[
        "origem_temperatura"
    ]
    == "observado_inmet"
)

temperatura_final.loc[
    mascara_observada,
    "qualidade_temperatura"
] = "observado"

In [132]:
print(
    "Dimensão:"
)

print(
    temperatura_final.shape
)


print(
    "\nMunicípios:"
)

print(
    temperatura_final[
        "codigo_ibge"
    ].nunique()
)


print(
    "\nAnos:"
)

print(
    temperatura_final[
        "ano"
    ].nunique()
)


print(
    "\nDuplicidades:"
)

print(
    temperatura_final[
        [
            "codigo_ibge",
            "ano"
        ]
    ]
    .duplicated()
    .sum()
)


print(
    "\nTemperatura final ausente:"
)

print(
    temperatura_final[
        "temperatura_final_c"
    ]
    .isna()
    .sum()
)

Dimensão:
(9966, 15)

Municípios:
1661

Anos:
6

Duplicidades:
0

Temperatura final ausente:
0


In [133]:
display(
    temperatura_final[
        "origem_temperatura"
    ]
    .value_counts()
)

display(
    temperatura_final[
        "qualidade_temperatura"
    ]
    .value_counts()
)

origem_temperatura
estimado_idw       9191
observado_inmet     775
Name: count, dtype: int64

qualidade_temperatura
alta           8365
observado       775
media           393
baixa           290
muito_baixa     143
Name: count, dtype: int64

In [134]:
display(
    temperatura_final[
        "temperatura_final_c"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

count    9966.000000
mean       20.986132
std         2.870842
min        10.634183
1%         15.143919
5%         17.050469
25%        18.763334
50%        20.282317
75%        23.595863
95%        25.684478
99%        26.643775
max        29.354010
Name: temperatura_final_c, dtype: float64

In [135]:
ARQUIVO_TEMPERATURA_CURATED = (
    CURATED_MUNICIPIO_ANO_DIR
    / "inmet_temperatura_municipio_ano_2019_2024.csv"
)

temperatura_final.to_csv(
    ARQUIVO_TEMPERATURA_CURATED,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Camada municipal de temperatura salva:"
)

print(
    ARQUIVO_TEMPERATURA_CURATED
)

Camada municipal de temperatura salva:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\inmet\municipio_ano\inmet_temperatura_municipio_ano_2019_2024.csv


## 5.9 — Estimativa espacial da umidade relativa do ar

A mesma metodologia espacial validada para precipitação e temperatura será
aplicada à umidade relativa média anual.

Serão utilizadas somente estações-ano com pelo menos 80% de cobertura válida
para a variável de umidade.

A metodologia utiliza:

- até 3 estações válidas mais próximas;
- raio preferencial de 200 km;
- raio expandido de 300 km;
- IDW com potência 2;
- prioridade para observações válidas existentes no próprio município.

A qualidade espacial da estimativa será registrada separadamente.

In [136]:
estacoes_umidade_validas = (
    estacoes_metrica[
        (
            estacoes_metrica[
                "cobertura_umidade_pct"
            ] >= COBERTURA_MINIMA
        )
        &
        (
            estacoes_metrica[
                "umidade_media_anual_pct"
            ].notna()
        )
    ]
    .copy()
)

print(
    "Estações-ano válidas para umidade:"
)

print(
    len(
        estacoes_umidade_validas
    )
)

display(
    estacoes_umidade_validas[
        "ano"
    ]
    .value_counts()
    .sort_index()
)

Estações-ano válidas para umidade:
751


ano
2019    153
2020    130
2021     92
2022     94
2023    147
2024    135
Name: count, dtype: int64

In [137]:
resumo_rede_umidade = (
    estacoes_umidade_validas
    .groupby(
        "ano"
    )
    .agg(
        estacoes_validas=(
            "codigo_wmo",
            "nunique"
        ),

        municipios_com_estacao=(
            "codigo_ibge",
            "nunique"
        )
    )
    .reset_index()
)

display(
    resumo_rede_umidade
)

,ano,estacoes_validas,municipios_com_estacao
0,2019,153,147
1,2020,130,125
2,2021,92,87
3,2022,94,89
4,2023,147,140
5,2024,135,128


In [138]:
diagnosticos_umidade = []

for ano in sorted(
    estacoes_umidade_validas[
        "ano"
    ].unique()
):

    print(
        f"Calculando umidade - {ano}..."
    )

    diagnostico_ano = (
        calcular_estacoes_proximas(
            municipios=municipios_pontos,
            estacoes=estacoes_umidade_validas,
            ano=int(ano),
            coluna_valor="umidade_media_anual_pct",
            coluna_cobertura="cobertura_umidade_pct",
            k=3
        )
    )

    diagnosticos_umidade.append(
        diagnostico_ano
    )


diagnostico_umidade = pd.concat(
    diagnosticos_umidade,
    ignore_index=True
)

print(
    "\nDimensão:"
)

print(
    diagnostico_umidade.shape
)

Calculando umidade - 2019...
Calculando umidade - 2020...
Calculando umidade - 2021...
Calculando umidade - 2022...
Calculando umidade - 2023...
Calculando umidade - 2024...

Dimensão:
(9966, 23)


In [139]:
resultado_idw_umidade = (
    diagnostico_umidade
    .apply(
        lambda linha:
            estimar_variavel_linha(
                linha,
                nome_saida=
                    "umidade_estimada_pct",

                potencia=
                    POTENCIA_IDW,

                raio_preferencial=
                    RAIO_PREFERENCIAL_KM,

                raio_expandido=
                    RAIO_EXPANDIDO_KM
            ),
        axis=1
    )
)

In [140]:
umidade_estimativa = pd.concat(
    [
        diagnostico_umidade[
            [
                "codigo_ibge",
                "municipio",
                "uf",
                "regiao",
                "ano"
            ]
        ]
        .reset_index(
            drop=True
        ),

        resultado_idw_umidade
        .reset_index(
            drop=True
        )
    ],
    axis=1
)

print(
    umidade_estimativa.shape
)

display(
    umidade_estimativa.head()
)

(9966, 9)


,codigo_ibge,municipio,uf,regiao,ano,umidade_estimada_pct,numero_estacoes_estimativa,distancia_max_estacoes_km,qualidade_espacial_estimativa
0,5219902,São Francisco de Goiás,GO,Centro-oeste,2019,60.271288,3,94.946721,alta
1,4316204,Rondinha,RS,Sul,2019,76.405028,3,65.019557,alta
2,4317558,Santo Antônio do Palma,RS,Sul,2019,78.396824,3,57.274133,alta
3,4209508,Laurentino,SC,Sul,2019,83.134326,3,86.066456,alta
4,4202107,Barra Velha,SC,Sul,2019,83.794815,3,96.733362,alta


In [141]:
umidade_observada = (
    inmet_municipio_ano_observado[
        [
            "codigo_ibge",
            "ano",
            "umidade_media_anual_pct",
            "numero_estacoes_umidade_validas",
            "cobertura_umidade_media_pct"
        ]
    ]
    .copy()
)

In [142]:
umidade_final = (
    umidade_estimativa
    .merge(
        umidade_observada,
        on=[
            "codigo_ibge",
            "ano"
        ],
        how="left"
    )
)

In [143]:
umidade_final[
    "umidade_final_pct"
] = (
    umidade_final[
        "umidade_media_anual_pct"
    ]
    .combine_first(
        umidade_final[
            "umidade_estimada_pct"
        ]
    )
)

In [144]:
umidade_final[
    "origem_umidade"
] = np.where(
    umidade_final[
        "umidade_media_anual_pct"
    ].notna(),

    "observado_inmet",

    "estimado_idw"
)

In [145]:
umidade_final[
    "qualidade_umidade"
] = (
    umidade_final[
        "qualidade_espacial_estimativa"
    ]
)

mascara_observada_umidade = (
    umidade_final[
        "origem_umidade"
    ]
    == "observado_inmet"
)

umidade_final.loc[
    mascara_observada_umidade,
    "qualidade_umidade"
] = "observado"

In [146]:
print(
    "Dimensão:"
)

print(
    umidade_final.shape
)


print(
    "\nMunicípios:"
)

print(
    umidade_final[
        "codigo_ibge"
    ].nunique()
)


print(
    "\nAnos:"
)

print(
    umidade_final[
        "ano"
    ].nunique()
)


print(
    "\nDuplicidades:"
)

print(
    umidade_final[
        [
            "codigo_ibge",
            "ano"
        ]
    ]
    .duplicated()
    .sum()
)


print(
    "\nUmidade final ausente:"
)

print(
    umidade_final[
        "umidade_final_pct"
    ]
    .isna()
    .sum()
)

Dimensão:
(9966, 15)

Municípios:
1661

Anos:
6

Duplicidades:
0

Umidade final ausente:
0


In [147]:
display(
    umidade_final[
        "origem_umidade"
    ]
    .value_counts()
)

origem_umidade
estimado_idw       9250
observado_inmet     716
Name: count, dtype: int64

In [148]:
display(
    umidade_final[
        "qualidade_umidade"
    ]
    .value_counts()
)

qualidade_umidade
alta           8341
observado       716
media           429
baixa           324
muito_baixa     156
Name: count, dtype: int64

In [149]:
display(
    umidade_final[
        "umidade_final_pct"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

count    9966.000000
mean       72.348863
std         7.041077
min        53.208944
1%         58.295983
5%         61.081903
25%        66.538527
50%        73.163060
75%        77.491446
95%        83.579444
99%        86.591000
max        92.745039
Name: umidade_final_pct, dtype: float64

In [150]:
print(
    "Umidade < 0%:"
)

print(
    (
        umidade_final[
            "umidade_final_pct"
        ] < 0
    ).sum()
)


print(
    "\nUmidade > 100%:"
)

print(
    (
        umidade_final[
            "umidade_final_pct"
        ] > 100
    ).sum()
)

Umidade < 0%:
0

Umidade > 100%:
0


In [151]:
print(
    "Menores umidades:"
)

display(
    umidade_final[
        [
            "municipio",
            "uf",
            "ano",
            "umidade_final_pct",
            "origem_umidade",
            "qualidade_umidade"
        ]
    ]
    .sort_values(
        "umidade_final_pct"
    )
    .head(20)
)


print(
    "\nMaiores umidades:"
)

display(
    umidade_final[
        [
            "municipio",
            "uf",
            "ano",
            "umidade_final_pct",
            "origem_umidade",
            "qualidade_umidade"
        ]
    ]
    .sort_values(
        "umidade_final_pct",
        ascending=False
    )
    .head(20)
)

Menores umidades:


,municipio,uf,ano,umidade_final_pct,origem_umidade,qualidade_umidade
9492,Cuiabá,MT,2024,53.208944,observado_inmet,observado
1126,Posse,GO,2019,53.392639,observado_inmet,observado
1026,Guarani de Goiás,GO,2019,53.616045,estimado_idw,media
1148,Mambaí,GO,2019,53.682337,estimado_idw,media
669,Buritinópolis,GO,2019,54.005435,estimado_idw,alta
1190,Iaciara,GO,2019,54.427956,estimado_idw,alta
2848,Cuiabá,MT,2020,54.466075,observado_inmet,observado
929,Simolândia,GO,2019,54.481211,estimado_idw,alta
72,Damianópolis,GO,2019,54.637078,estimado_idw,alta
9022,Várzea Grande,MT,2024,54.959967,estimado_idw,alta



Maiores umidades:


,municipio,uf,ano,umidade_final_pct,origem_umidade,qualidade_umidade
777,Rancho Queimado,SC,2019,92.745039,observado_inmet,observado
1381,Águas Mornas,SC,2019,90.763056,estimado_idw,alta
1090,Angelina,SC,2019,90.470628,estimado_idw,alta
1398,Anitápolis,SC,2019,89.521126,estimado_idw,alta
7046,Rio Negrinho,SC,2023,89.338532,observado_inmet,observado
704,Alfredo Wagner,SC,2019,89.019896,estimado_idw,alta
402,Rio Negrinho,SC,2019,88.987942,observado_inmet,observado
1510,Morretes,PR,2019,88.960616,observado_inmet,observado
1223,Piên,PR,2019,88.806628,estimado_idw,alta
6965,São Bento do Sul,SC,2023,88.752566,estimado_idw,alta


In [152]:
ARQUIVO_UMIDADE_CURATED = (
    CURATED_MUNICIPIO_ANO_DIR
    / "inmet_umidade_municipio_ano_2019_2024.csv"
)

umidade_final.to_csv(
    ARQUIVO_UMIDADE_CURATED,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Camada municipal de umidade salva:"
)

print(
    ARQUIVO_UMIDADE_CURATED
)

Camada municipal de umidade salva:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\inmet\municipio_ano\inmet_umidade_municipio_ano_2019_2024.csv


## 5.10 — Consolidação final da camada climática municipal

As etapas de precipitação, temperatura e umidade foram concluídas
individualmente.

Cada variável foi construída considerando:

- observações INMET com cobertura mínima de 80%;
- associação espacial das estações à Malha Municipal IBGE 2024;
- prioridade para observações válidas existentes no município;
- estimativa espacial por IDW quando não havia observação municipal válida;
- até três estações meteorológicas;
- raio preferencial de 200 km;
- raio expandido de 300 km;
- potência IDW igual a 2;
- classificação da qualidade espacial das estimativas.

Nesta etapa as três variáveis serão consolidadas em uma única base:

`município × ano`

A base resultante será o dataset CURATED oficial do INMET utilizado
nas integrações posteriores do projeto AgroESG.

In [153]:
precipitacao_curated = (
    precipitacao_final[
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao",
            "ano",
            "precipitacao_final_mm",
            "origem_precipitacao",
            "qualidade_precipitacao",
            "numero_estacoes_estimativa",
            "distancia_max_estacoes_km",
            "numero_estacoes_precipitacao_validas",
            "cobertura_precipitacao_media_pct"
        ]
    ]
    .copy()
)

In [154]:
precipitacao_curated = (
    precipitacao_curated
    .rename(
        columns={
            "precipitacao_final_mm":
                "precipitacao_anual_mm",

            "qualidade_precipitacao":
                "qualidade_espacial_precipitacao",

            "numero_estacoes_estimativa":
                "numero_estacoes_idw_precipitacao",

            "distancia_max_estacoes_km":
                "distancia_max_idw_precipitacao_km",

            "numero_estacoes_precipitacao_validas":
                "numero_estacoes_observadas_precipitacao",

            "cobertura_precipitacao_media_pct":
                "cobertura_observada_precipitacao_pct"
        }
    )
)

In [155]:
mascara_precip_observada = (
    precipitacao_curated[
        "origem_precipitacao"
    ] == "observado_inmet"
)

precipitacao_curated.loc[
    mascara_precip_observada,
    "numero_estacoes_idw_precipitacao"
] = np.nan

precipitacao_curated.loc[
    mascara_precip_observada,
    "distancia_max_idw_precipitacao_km"
] = np.nan

In [156]:
temperatura_curated = (
    temperatura_final[
        [
            "codigo_ibge",
            "ano",
            "temperatura_final_c",
            "origem_temperatura",
            "qualidade_temperatura",
            "numero_estacoes_estimativa",
            "distancia_max_estacoes_km",
            "numero_estacoes_temperatura_validas",
            "cobertura_temperatura_media_pct"
        ]
    ]
    .copy()
)

In [ ]:
temperatura_curated = (
    temperatura_curated
    .rename(
        columns={
            "temperatura_final_c":
                "temperatura_media_anual_c",

            "qualidade_temperatura":
                "qualidade_espacial_temperatura",

            "numero_estacoes_estimativa":
                "numero_estacoes_idw_temperatura",

            "distancia_max_estacoes_km":
                "distancia_max_idw_temperatura_km",

            "numero_estacoes_temperatura_validas":
                "numero_estacoes_observadas_temperatura",

            "cobertura_temperatura_media_pct":
                "cobertura_observada_temperatura_pct"
        }
    )
)

In [157]:
mascara_temp_observada = (
    temperatura_curated[
        "origem_temperatura"
    ] == "observado_inmet"
)

temperatura_curated.loc[
    mascara_temp_observada,
    "numero_estacoes_idw_temperatura"
] = np.nan

temperatura_curated.loc[
    mascara_temp_observada,
    "distancia_max_idw_temperatura_km"
] = np.nan

In [158]:
umidade_curated = (
    umidade_final[
        [
            "codigo_ibge",
            "ano",
            "umidade_final_pct",
            "origem_umidade",
            "qualidade_umidade",
            "numero_estacoes_estimativa",
            "distancia_max_estacoes_km",
            "numero_estacoes_umidade_validas",
            "cobertura_umidade_media_pct"
        ]
    ]
    .copy()
)

In [159]:
umidade_curated = (
    umidade_curated
    .rename(
        columns={
            "umidade_final_pct":
                "umidade_media_anual_pct",

            "qualidade_umidade":
                "qualidade_espacial_umidade",

            "numero_estacoes_estimativa":
                "numero_estacoes_idw_umidade",

            "distancia_max_estacoes_km":
                "distancia_max_idw_umidade_km",

            "numero_estacoes_umidade_validas":
                "numero_estacoes_observadas_umidade",

            "cobertura_umidade_media_pct":
                "cobertura_observada_umidade_pct"
        }
    )
)

In [160]:
mascara_umidade_observada = (
    umidade_curated[
        "origem_umidade"
    ] == "observado_inmet"
)

umidade_curated.loc[
    mascara_umidade_observada,
    "numero_estacoes_idw_umidade"
] = np.nan

umidade_curated.loc[
    mascara_umidade_observada,
    "distancia_max_idw_umidade_km"
] = np.nan

In [161]:
inmet_municipio_ano_final = (
    precipitacao_curated
    .merge(
        temperatura_curated,
        on=[
            "codigo_ibge",
            "ano"
        ],
        how="inner",
        validate="one_to_one"
    )
)

In [162]:
inmet_municipio_ano_final = (
    inmet_municipio_ano_final
    .merge(
        umidade_curated,
        on=[
            "codigo_ibge",
            "ano"
        ],
        how="inner",
        validate="one_to_one"
    )
)

In [163]:
print(
    "Dimensão consolidada:"
)

print(
    inmet_municipio_ano_final.shape
)

display(
    inmet_municipio_ano_final.head()
)

Dimensão consolidada:
(9966, 28)


,codigo_ibge,municipio,uf,regiao,ano,precipitacao_anual_mm,origem_precipitacao,qualidade_espacial_precipitacao,numero_estacoes_idw_precipitacao,distancia_max_idw_precipitacao_km,...,cobertura_temperatura_media_pct,numero_estacoes_idw_temperatura,distancia_max_idw_temperatura_km,umidade_media_anual_pct,origem_umidade,qualidade_espacial_umidade,numero_estacoes_idw_umidade,distancia_max_idw_umidade_km,numero_estacoes_observadas_umidade,cobertura_observada_umidade_pct
0,5219902,São Francisco de Goiás,GO,Centro-oeste,2019,1073.612878,estimado_idw,alta,3.0,94.946721,...,NaN,NaN,NaN,60.271288,estimado_idw,alta,3.0,94.946721,NaN,NaN
1,4316204,Rondinha,RS,Sul,2019,1606.290803,estimado_idw,alta,3.0,65.019557,...,NaN,NaN,NaN,76.405028,estimado_idw,alta,3.0,65.019557,NaN,NaN
2,4317558,Santo Antônio do Palma,RS,Sul,2019,1667.721094,estimado_idw,alta,3.0,57.274133,...,NaN,NaN,NaN,78.396824,estimado_idw,alta,3.0,57.274133,NaN,NaN
3,4209508,Laurentino,SC,Sul,2019,1302.575419,estimado_idw,alta,3.0,86.066456,...,NaN,NaN,NaN,83.134326,estimado_idw,alta,3.0,86.066456,NaN,NaN
4,4202107,Barra Velha,SC,Sul,2019,1528.704529,estimado_idw,alta,3.0,96.733362,...,NaN,NaN,NaN,83.794815,estimado_idw,alta,3.0,96.733362,NaN,NaN


In [164]:
inmet_municipio_ano_final[
    "regiao"
] = (
    inmet_municipio_ano_final[
        "regiao"
    ]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(
        "-",
        "_",
        regex=False
    )
)

In [165]:
display(
    inmet_municipio_ano_final[
        "regiao"
    ]
    .value_counts()
)

regiao
sul             7158
centro_oeste    2808
Name: count, dtype: Int64

In [166]:
inmet_municipio_ano_final[
    "quantidade_variaveis_observadas"
] = (
    (
        inmet_municipio_ano_final[
            "origem_precipitacao"
        ] == "observado_inmet"
    ).astype(int)
    +
    (
        inmet_municipio_ano_final[
            "origem_temperatura"
        ] == "observado_inmet"
    ).astype(int)
    +
    (
        inmet_municipio_ano_final[
            "origem_umidade"
        ] == "observado_inmet"
    ).astype(int)
)

In [167]:
display(
    inmet_municipio_ano_final[
        "quantidade_variaveis_observadas"
    ]
    .value_counts()
    .sort_index()
)

quantidade_variaveis_observadas
0    9188
1      10
2     139
3     629
Name: count, dtype: int64

In [168]:
inmet_municipio_ano_final[
    "tipo_representacao_climatica"
] = np.select(
    [
        (
            inmet_municipio_ano_final[
                "quantidade_variaveis_observadas"
            ] == 3
        ),

        (
            inmet_municipio_ano_final[
                "quantidade_variaveis_observadas"
            ].between(
                1,
                2
            )
        )
    ],

    [
        "observado_tres_variaveis",
        "misto_observado_estimado"
    ],

    default="estimado_tres_variaveis"
)

In [169]:
display(
    inmet_municipio_ano_final[
        "tipo_representacao_climatica"
    ]
    .value_counts()
)

tipo_representacao_climatica
estimado_tres_variaveis     9188
observado_tres_variaveis     629
misto_observado_estimado     149
Name: count, dtype: int64

In [170]:
MAPA_QUALIDADE_SCORE = {
    "muito_baixa": 1,
    "baixa": 2,
    "media": 3,
    "alta": 4,
    "observado": 5
}

In [171]:
scores_qualidade = pd.DataFrame(
    {
        "precipitacao":
            inmet_municipio_ano_final[
                "qualidade_espacial_precipitacao"
            ]
            .map(
                MAPA_QUALIDADE_SCORE
            ),

        "temperatura":
            inmet_municipio_ano_final[
                "qualidade_espacial_temperatura"
            ]
            .map(
                MAPA_QUALIDADE_SCORE
            ),

        "umidade":
            inmet_municipio_ano_final[
                "qualidade_espacial_umidade"
            ]
            .map(
                MAPA_QUALIDADE_SCORE
            )
    }
)

KeyError: 'qualidade_espacial_temperatura'

In [172]:
print("Colunas relacionadas à temperatura:")

for coluna in inmet_municipio_ano_final.columns:

    if "temperatura" in coluna:
        print(coluna)


print("\nColunas relacionadas à umidade:")

for coluna in inmet_municipio_ano_final.columns:

    if "umidade" in coluna:
        print(coluna)


print("\nColunas relacionadas à precipitação:")

for coluna in inmet_municipio_ano_final.columns:

    if "precipitacao" in coluna:
        print(coluna)

Colunas relacionadas à temperatura:
temperatura_final_c
origem_temperatura
qualidade_temperatura
numero_estacoes_temperatura_validas
cobertura_temperatura_media_pct
numero_estacoes_idw_temperatura
distancia_max_idw_temperatura_km

Colunas relacionadas à umidade:
umidade_media_anual_pct
origem_umidade
qualidade_espacial_umidade
numero_estacoes_idw_umidade
distancia_max_idw_umidade_km
numero_estacoes_observadas_umidade
cobertura_observada_umidade_pct

Colunas relacionadas à precipitação:
precipitacao_anual_mm
origem_precipitacao
qualidade_espacial_precipitacao
numero_estacoes_idw_precipitacao
distancia_max_idw_precipitacao_km
numero_estacoes_observadas_precipitacao
cobertura_observada_precipitacao_pct


In [173]:
# ============================================================
# CORREÇÃO / PADRONIZAÇÃO DOS NOMES DE COLUNAS
# DA BASE INMET CONSOLIDADA
# ============================================================

mapa_renomear_final = {

    # --------------------------------------------------------
    # Precipitação
    # --------------------------------------------------------

    "qualidade_precipitacao":
        "qualidade_espacial_precipitacao",

    "numero_estacoes_estimativa_precipitacao":
        "numero_estacoes_idw_precipitacao",

    "distancia_max_estacoes_precipitacao_km":
        "distancia_max_idw_precipitacao_km",

    "numero_estacoes_precipitacao_validas":
        "numero_estacoes_observadas_precipitacao",

    "cobertura_precipitacao_media_pct":
        "cobertura_observada_precipitacao_pct",


    # --------------------------------------------------------
    # Temperatura
    # --------------------------------------------------------

    "qualidade_temperatura":
        "qualidade_espacial_temperatura",

    "numero_estacoes_temperatura_validas":
        "numero_estacoes_observadas_temperatura",

    "cobertura_temperatura_media_pct":
        "cobertura_observada_temperatura_pct",


    # --------------------------------------------------------
    # Umidade
    # --------------------------------------------------------

    "qualidade_umidade":
        "qualidade_espacial_umidade",

    "numero_estacoes_umidade_validas":
        "numero_estacoes_observadas_umidade",

    "cobertura_umidade_media_pct":
        "cobertura_observada_umidade_pct"
}


for coluna_antiga, coluna_nova in mapa_renomear_final.items():

    if (
        coluna_antiga
        in inmet_municipio_ano_final.columns
        and
        coluna_nova
        not in inmet_municipio_ano_final.columns
    ):

        inmet_municipio_ano_final = (
            inmet_municipio_ano_final
            .rename(
                columns={
                    coluna_antiga:
                        coluna_nova
                }
            )
        )


print(
    "Padronização de nomes concluída."
)

Padronização de nomes concluída.


In [174]:
colunas_qualidade_esperadas = [
    "qualidade_espacial_precipitacao",
    "qualidade_espacial_temperatura",
    "qualidade_espacial_umidade"
]

for coluna in colunas_qualidade_esperadas:

    print(
        coluna,
        "->",
        coluna in inmet_municipio_ano_final.columns
    )

qualidade_espacial_precipitacao -> True
qualidade_espacial_temperatura -> True
qualidade_espacial_umidade -> True


In [175]:
colunas_rastreabilidade_esperadas = [

    # precipitação
    "numero_estacoes_observadas_precipitacao",
    "cobertura_observada_precipitacao_pct",
    "numero_estacoes_idw_precipitacao",
    "distancia_max_idw_precipitacao_km",

    # temperatura
    "numero_estacoes_observadas_temperatura",
    "cobertura_observada_temperatura_pct",
    "numero_estacoes_idw_temperatura",
    "distancia_max_idw_temperatura_km",

    # umidade
    "numero_estacoes_observadas_umidade",
    "cobertura_observada_umidade_pct",
    "numero_estacoes_idw_umidade",
    "distancia_max_idw_umidade_km"
]


for coluna in colunas_rastreabilidade_esperadas:

    print(
        coluna,
        "->",
        coluna in inmet_municipio_ano_final.columns
    )

numero_estacoes_observadas_precipitacao -> True
cobertura_observada_precipitacao_pct -> True
numero_estacoes_idw_precipitacao -> True
distancia_max_idw_precipitacao_km -> True
numero_estacoes_observadas_temperatura -> True
cobertura_observada_temperatura_pct -> True
numero_estacoes_idw_temperatura -> True
distancia_max_idw_temperatura_km -> True
numero_estacoes_observadas_umidade -> True
cobertura_observada_umidade_pct -> True
numero_estacoes_idw_umidade -> True
distancia_max_idw_umidade_km -> True


In [176]:
MAPA_QUALIDADE_SCORE = {
    "muito_baixa": 1,
    "baixa": 2,
    "media": 3,
    "alta": 4,
    "observado": 5
}

In [177]:
scores_qualidade = pd.DataFrame(
    {
        "precipitacao":
            inmet_municipio_ano_final[
                "qualidade_espacial_precipitacao"
            ]
            .map(
                MAPA_QUALIDADE_SCORE
            ),

        "temperatura":
            inmet_municipio_ano_final[
                "qualidade_espacial_temperatura"
            ]
            .map(
                MAPA_QUALIDADE_SCORE
            ),

        "umidade":
            inmet_municipio_ano_final[
                "qualidade_espacial_umidade"
            ]
            .map(
                MAPA_QUALIDADE_SCORE
            )
    }
)

In [178]:
display(
    scores_qualidade.head()
)

print(
    "\nValores ausentes nos scores:"
)

print(
    scores_qualidade
    .isna()
    .sum()
)

,precipitacao,temperatura,umidade
0,4,4,4
1,4,4,4
2,4,4,4
3,4,4,4
4,4,4,4



Valores ausentes nos scores:
precipitacao    0
temperatura     0
umidade         0
dtype: int64


In [179]:
inmet_municipio_ano_final[
    "score_qualidade_climatica"
] = (
    scores_qualidade
    .min(
        axis=1
    )
)

In [180]:
MAPA_SCORE_QUALIDADE = {
    1: "muito_baixa",
    2: "baixa",
    3: "media",
    4: "alta",
    5: "observado"
}


inmet_municipio_ano_final[
    "qualidade_climatica_geral"
] = (
    inmet_municipio_ano_final[
        "score_qualidade_climatica"
    ]
    .map(
        MAPA_SCORE_QUALIDADE
    )
)

In [181]:
display(
    inmet_municipio_ano_final[
        "qualidade_climatica_geral"
    ]
    .value_counts()
)

qualidade_climatica_geral
alta           8119
observado       629
media           514
baixa           500
muito_baixa     204
Name: count, dtype: int64

In [182]:
print(
    "Score climático ausente:"
)

print(
    inmet_municipio_ano_final[
        "score_qualidade_climatica"
    ]
    .isna()
    .sum()
)


print(
    "\nQualidade climática geral ausente:"
)

print(
    inmet_municipio_ano_final[
        "qualidade_climatica_geral"
    ]
    .isna()
    .sum()
)

Score climático ausente:
0

Qualidade climática geral ausente:
0


In [183]:
if (
    "temperatura_final_c"
    in inmet_municipio_ano_final.columns
):

    inmet_municipio_ano_final = (
        inmet_municipio_ano_final
        .rename(
            columns={
                "temperatura_final_c":
                    "temperatura_media_anual_c"
            }
        )
    )

print(
    "temperatura_media_anual_c"
    in inmet_municipio_ano_final.columns
)

True


In [184]:
colunas_climaticas_principais = [
    "precipitacao_anual_mm",
    "temperatura_media_anual_c",
    "umidade_media_anual_pct"
]

for coluna in colunas_climaticas_principais:

    print(
        coluna,
        "->",
        coluna in inmet_municipio_ano_final.columns
    )

precipitacao_anual_mm -> True
temperatura_media_anual_c -> True
umidade_media_anual_pct -> True


In [185]:
COLUNAS_FINAIS_INMET = [

    # ========================================================
    # IDENTIFICAÇÃO
    # ========================================================

    "codigo_ibge",
    "municipio",
    "uf",
    "regiao",
    "ano",


    # ========================================================
    # PRECIPITAÇÃO
    # ========================================================

    "precipitacao_anual_mm",
    "origem_precipitacao",
    "qualidade_espacial_precipitacao",

    "numero_estacoes_observadas_precipitacao",
    "cobertura_observada_precipitacao_pct",

    "numero_estacoes_idw_precipitacao",
    "distancia_max_idw_precipitacao_km",


    # ========================================================
    # TEMPERATURA
    # ========================================================

    "temperatura_media_anual_c",
    "origem_temperatura",
    "qualidade_espacial_temperatura",

    "numero_estacoes_observadas_temperatura",
    "cobertura_observada_temperatura_pct",

    "numero_estacoes_idw_temperatura",
    "distancia_max_idw_temperatura_km",


    # ========================================================
    # UMIDADE
    # ========================================================

    "umidade_media_anual_pct",
    "origem_umidade",
    "qualidade_espacial_umidade",

    "numero_estacoes_observadas_umidade",
    "cobertura_observada_umidade_pct",

    "numero_estacoes_idw_umidade",
    "distancia_max_idw_umidade_km",


    # ========================================================
    # QUALIDADE GERAL
    # ========================================================

    "quantidade_variaveis_observadas",
    "tipo_representacao_climatica",
    "score_qualidade_climatica",
    "qualidade_climatica_geral"
]

In [186]:
colunas_ausentes = [
    coluna
    for coluna in COLUNAS_FINAIS_INMET
    if coluna
    not in inmet_municipio_ano_final.columns
]

print(
    "Colunas finais ausentes:"
)

print(
    colunas_ausentes
)

Colunas finais ausentes:
[]


In [187]:
inmet_municipio_ano_final = (
    inmet_municipio_ano_final[
        COLUNAS_FINAIS_INMET
    ]
    .copy()
)

print(
    inmet_municipio_ano_final.shape
)

display(
    inmet_municipio_ano_final.head()
)

(9966, 30)


,codigo_ibge,municipio,uf,regiao,ano,precipitacao_anual_mm,origem_precipitacao,qualidade_espacial_precipitacao,numero_estacoes_observadas_precipitacao,cobertura_observada_precipitacao_pct,...,origem_umidade,qualidade_espacial_umidade,numero_estacoes_observadas_umidade,cobertura_observada_umidade_pct,numero_estacoes_idw_umidade,distancia_max_idw_umidade_km,quantidade_variaveis_observadas,tipo_representacao_climatica,score_qualidade_climatica,qualidade_climatica_geral
0,5219902,São Francisco de Goiás,GO,centro_oeste,2019,1073.612878,estimado_idw,alta,NaN,NaN,...,estimado_idw,alta,NaN,NaN,3.0,94.946721,0,estimado_tres_variaveis,4,alta
1,4316204,Rondinha,RS,sul,2019,1606.290803,estimado_idw,alta,NaN,NaN,...,estimado_idw,alta,NaN,NaN,3.0,65.019557,0,estimado_tres_variaveis,4,alta
2,4317558,Santo Antônio do Palma,RS,sul,2019,1667.721094,estimado_idw,alta,NaN,NaN,...,estimado_idw,alta,NaN,NaN,3.0,57.274133,0,estimado_tres_variaveis,4,alta
3,4209508,Laurentino,SC,sul,2019,1302.575419,estimado_idw,alta,NaN,NaN,...,estimado_idw,alta,NaN,NaN,3.0,86.066456,0,estimado_tres_variaveis,4,alta
4,4202107,Barra Velha,SC,sul,2019,1528.704529,estimado_idw,alta,NaN,NaN,...,estimado_idw,alta,NaN,NaN,3.0,96.733362,0,estimado_tres_variaveis,4,alta


In [188]:
print(
    "=============================================="
)

print(
    "VALIDAÇÃO FINAL — INMET CURATED"
)

print(
    "=============================================="
)


print(
    "\nDimensão:"
)

print(
    inmet_municipio_ano_final.shape
)


print(
    "\nMunicípios:"
)

print(
    inmet_municipio_ano_final[
        "codigo_ibge"
    ].nunique()
)


print(
    "\nPeríodo:"
)

print(
    inmet_municipio_ano_final[
        "ano"
    ].min(),
    "até",
    inmet_municipio_ano_final[
        "ano"
    ].max()
)


print(
    "\nDuplicidades codigo_ibge + ano:"
)

print(
    inmet_municipio_ano_final[
        [
            "codigo_ibge",
            "ano"
        ]
    ]
    .duplicated()
    .sum()
)


print(
    "\nPrecipitação ausente:"
)

print(
    inmet_municipio_ano_final[
        "precipitacao_anual_mm"
    ]
    .isna()
    .sum()
)


print(
    "\nTemperatura ausente:"
)

print(
    inmet_municipio_ano_final[
        "temperatura_media_anual_c"
    ]
    .isna()
    .sum()
)


print(
    "\nUmidade ausente:"
)

print(
    inmet_municipio_ano_final[
        "umidade_media_anual_pct"
    ]
    .isna()
    .sum()
)

VALIDAÇÃO FINAL — INMET CURATED

Dimensão:
(9966, 30)

Municípios:
1661

Período:
2019 até 2024

Duplicidades codigo_ibge + ano:
0

Precipitação ausente:
0

Temperatura ausente:
0

Umidade ausente:
0


In [189]:
display(
    inmet_municipio_ano_final[
        "ano"
    ]
    .value_counts()
    .sort_index()
)

ano
2019    1661
2020    1661
2021    1661
2022    1661
2023    1661
2024    1661
Name: count, dtype: int64

In [190]:
print(
    inmet_municipio_ano_final[
        "ano"
    ]
    .value_counts()
    .sum()
)

9966


In [191]:
print(
    "Códigos IBGE diferentes de 7 caracteres:"
)

print(
    (
        inmet_municipio_ano_final[
            "codigo_ibge"
        ]
        .astype(str)
        .str.len()
        != 7
    ).sum()
)

Códigos IBGE diferentes de 7 caracteres:
0


In [192]:
display(
    inmet_municipio_ano_final[
        "uf"
    ]
    .value_counts()
    .sort_index()
)

uf
DF       6
GO    1476
MS     474
MT     852
PR    2394
RS    2994
SC    1770
Name: count, dtype: int64

In [193]:
print(
    "Precipitação negativa:"
)

print(
    (
        inmet_municipio_ano_final[
            "precipitacao_anual_mm"
        ] < 0
    ).sum()
)


print(
    "\nUmidade fora de 0–100%:"
)

print(
    (
        (
            inmet_municipio_ano_final[
                "umidade_media_anual_pct"
            ] < 0
        )
        |
        (
            inmet_municipio_ano_final[
                "umidade_media_anual_pct"
            ] > 100
        )
    ).sum()
)

Precipitação negativa:
0

Umidade fora de 0–100%:
0


In [194]:
print(
    "Representação climática:"
)

display(
    inmet_municipio_ano_final[
        "tipo_representacao_climatica"
    ]
    .value_counts()
)


print(
    "\nQualidade climática geral:"
)

display(
    inmet_municipio_ano_final[
        "qualidade_climatica_geral"
    ]
    .value_counts()
)

Representação climática:


tipo_representacao_climatica
estimado_tres_variaveis     9188
observado_tres_variaveis     629
misto_observado_estimado     149
Name: count, dtype: int64


Qualidade climática geral:


qualidade_climatica_geral
alta           8119
observado       629
media           514
baixa           500
muito_baixa     204
Name: count, dtype: int64

In [195]:
ARQUIVO_INMET_FINAL = (
    CURATED_MUNICIPIO_ANO_DIR
    / "inmet_municipio_ano_2019_2024.csv"
)

inmet_municipio_ano_final.to_csv(
    ARQUIVO_INMET_FINAL,
    index=False,
    encoding="utf-8-sig"
)

print(
    "=============================================="
)

print(
    "INMET CURATED FINAL SALVO"
)

print(
    "=============================================="
)

print(
    ARQUIVO_INMET_FINAL
)

INMET CURATED FINAL SALVO
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\inmet\municipio_ano\inmet_municipio_ano_2019_2024.csv


In [196]:
resumo_qualidade_final = (
    inmet_municipio_ano_final
    .groupby(
        [
            "ano",
            "qualidade_climatica_geral"
        ]
    )
    .size()
    .reset_index(
        name="registros"
    )
)

display(
    resumo_qualidade_final
)

,ano,qualidade_climatica_geral,registros
0,2019,alta,1441
1,2019,baixa,21
2,2019,media,54
3,2019,muito_baixa,9
4,2019,observado,136
5,2020,alta,1433
6,2020,baixa,51
7,2020,media,36
8,2020,muito_baixa,30
9,2020,observado,111


In [197]:
resumo_qualidade_final.to_csv(
    QUALITY_DIR
    / "resumo_qualidade_climatica_municipal_2019_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Resumo final de qualidade salvo."
)

Resumo final de qualidade salvo.


In [198]:
print(
    "Quantidade de colunas esperadas:"
)

print(
    len(
        COLUNAS_FINAIS_INMET
    )
)

print(
    "\nQuantidade de colunas da base:"
)

print(
    len(
        inmet_municipio_ano_final.columns
    )
)

Quantidade de colunas esperadas:
30

Quantidade de colunas da base:
30


In [199]:
print(
    "Lista exatamente igual?"
)

print(
    inmet_municipio_ano_final.columns.tolist()
    ==
    COLUNAS_FINAIS_INMET
)

Lista exatamente igual?
True


## Correção de metadados IDW da temperatura

Durante a auditoria para integração das bases Curated, foi identificado que
os campos auxiliares `numero_estacoes_idw_temperatura` e
`distancia_max_idw_temperatura_km` da base climática integrada estavam
totalmente nulos.

Os valores existem na base específica de temperatura. Esta etapa corrige
somente os metadados de rastreabilidade do IDW, sem recalcular ou alterar
temperatura, precipitação, umidade ou a interpolação espacial.

In [201]:
# ============================================================
# CORREÇÃO — METADADOS IDW DA TEMPERATURA
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


BASE_DIR = Path(
    r"C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao"
)


nome_temperatura = (
    "inmet_temperatura_municipio_ano_2019_2024.csv"
)

nome_integrado = (
    "inmet_municipio_ano_2019_2024.csv"
)


arquivos_temperatura = list(
    BASE_DIR.rglob(
        nome_temperatura
    )
)

arquivos_integrado = list(
    BASE_DIR.rglob(
        nome_integrado
    )
)


print(
    "Arquivos de temperatura encontrados:",
    len(arquivos_temperatura)
)

print(
    "Arquivos integrados encontrados:",
    len(arquivos_integrado)
)


print(
    "\nTemperatura:"
)

for arquivo in arquivos_temperatura:
    print(arquivo)


print(
    "\nIntegrado:"
)

for arquivo in arquivos_integrado:
    print(arquivo)

Arquivos de temperatura encontrados: 1
Arquivos integrados encontrados: 1

Temperatura:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\inmet\municipio_ano\inmet_temperatura_municipio_ano_2019_2024.csv

Integrado:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\inmet\municipio_ano\inmet_municipio_ano_2019_2024.csv


In [202]:
ARQUIVO_TEMPERATURA = (
    arquivos_temperatura[0]
)

ARQUIVO_INMET_INTEGRADO = (
    arquivos_integrado[0]
)


temperatura_correcao = pd.read_csv(
    ARQUIVO_TEMPERATURA,
    dtype={
        "codigo_ibge": "string"
    }
)


inmet_integrado_correcao = pd.read_csv(
    ARQUIVO_INMET_INTEGRADO,
    dtype={
        "codigo_ibge": "string"
    }
)


for base in [
    temperatura_correcao,
    inmet_integrado_correcao
]:

    base["codigo_ibge"] = (
        base["codigo_ibge"]
        .str.zfill(7)
    )

    base["ano"] = (
        pd.to_numeric(
            base["ano"],
            errors="raise"
        )
        .astype(int)
    )


print(
    "Temperatura:",
    temperatura_correcao.shape
)

print(
    "Integrado:",
    inmet_integrado_correcao.shape
)

Temperatura: (9966, 15)
Integrado: (9966, 30)


In [203]:
# ============================================================
# AUDITORIA ANTES DA CORREÇÃO
# ============================================================

CHAVE = [
    "codigo_ibge",
    "ano"
]


print(
    "Duplicatas temperatura:"
)

print(
    temperatura_correcao
    .duplicated(
        subset=CHAVE
    )
    .sum()
)


print(
    "\nDuplicatas integrado:"
)

print(
    inmet_integrado_correcao
    .duplicated(
        subset=CHAVE
    )
    .sum()
)


print(
    "\nOrigem da temperatura:"
)

display(
    temperatura_correcao[
        "origem_temperatura"
    ]
    .value_counts()
)


print(
    "\nNulos atuais no integrado:"
)

print(
    "numero_estacoes_idw_temperatura:",
    inmet_integrado_correcao[
        "numero_estacoes_idw_temperatura"
    ]
    .isna()
    .sum()
)

print(
    "distancia_max_idw_temperatura_km:",
    inmet_integrado_correcao[
        "distancia_max_idw_temperatura_km"
    ]
    .isna()
    .sum()
)

Duplicatas temperatura:
0

Duplicatas integrado:
0

Origem da temperatura:


origem_temperatura
estimado_idw       9191
observado_inmet     775
Name: count, dtype: int64


Nulos atuais no integrado:
numero_estacoes_idw_temperatura: 9966
distancia_max_idw_temperatura_km: 9966


In [204]:
# ============================================================
# CORREÇÃO DOS METADADOS IDW DE TEMPERATURA
# ============================================================

COLUNAS_ORIGINAIS = (
    inmet_integrado_correcao
    .columns
    .tolist()
)


metadados_temperatura = (
    temperatura_correcao[
        [
            "codigo_ibge",
            "ano",
            "origem_temperatura",
            "numero_estacoes_estimativa",
            "distancia_max_estacoes_km"
        ]
    ]
    .rename(
        columns={
            "origem_temperatura":
                "origem_temperatura_fonte",

            "numero_estacoes_estimativa":
                "numero_estacoes_estimativa_temperatura",

            "distancia_max_estacoes_km":
                "distancia_max_estimativa_temperatura_km"
        }
    )
)


inmet_integrado_corrigido = (
    inmet_integrado_correcao
    .merge(
        metadados_temperatura,
        on=[
            "codigo_ibge",
            "ano"
        ],
        how="left",
        validate="one_to_one"
    )
)


print(
    "Diferenças na origem da temperatura:"
)

print(
    (
        inmet_integrado_corrigido[
            "origem_temperatura"
        ]
        !=
        inmet_integrado_corrigido[
            "origem_temperatura_fonte"
        ]
    )
    .sum()
)

Diferenças na origem da temperatura:
0


In [205]:
mascara_temperatura_idw = (
    inmet_integrado_corrigido[
        "origem_temperatura"
    ]
    .eq(
        "estimado_idw"
    )
)


inmet_integrado_corrigido[
    "numero_estacoes_idw_temperatura"
] = (
    inmet_integrado_corrigido[
        "numero_estacoes_estimativa_temperatura"
    ]
    .where(
        mascara_temperatura_idw
    )
)


inmet_integrado_corrigido[
    "distancia_max_idw_temperatura_km"
] = (
    inmet_integrado_corrigido[
        "distancia_max_estimativa_temperatura_km"
    ]
    .where(
        mascara_temperatura_idw
    )
)


inmet_integrado_corrigido = (
    inmet_integrado_corrigido[
        COLUNAS_ORIGINAIS
    ]
    .copy()
)

In [206]:
# ============================================================
# VALIDAÇÃO DA CORREÇÃO
# ============================================================

print(
    "Dimensão:"
)

print(
    inmet_integrado_corrigido.shape
)


print(
    "\nDuplicatas:"
)

print(
    inmet_integrado_corrigido
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


print(
    "\nOrigem da temperatura:"
)

display(
    inmet_integrado_corrigido[
        "origem_temperatura"
    ]
    .value_counts()
)


print(
    "\nIDW temperatura preenchido por origem:"
)

display(
    inmet_integrado_corrigido
    .groupby(
        "origem_temperatura"
    )[
        [
            "numero_estacoes_idw_temperatura",
            "distancia_max_idw_temperatura_km"
        ]
    ]
    .count()
)


print(
    "\nNulos da variável climática principal:"
)

print(
    inmet_integrado_corrigido[
        "temperatura_media_anual_c"
    ]
    .isna()
    .sum()
)

Dimensão:
(9966, 30)

Duplicatas:
0

Origem da temperatura:


origem_temperatura
estimado_idw       9191
observado_inmet     775
Name: count, dtype: int64


IDW temperatura preenchido por origem:


,numero_estacoes_idw_temperatura,distancia_max_idw_temperatura_km
origem_temperatura,,
estimado_idw,9191,9191
observado_inmet,0,0



Nulos da variável climática principal:
0


In [207]:
# ============================================================
# GARANTIA — SOMENTE DUAS COLUNAS FORAM CORRIGIDAS
# ============================================================

COLUNAS_CORRIGIDAS = [
    "numero_estacoes_idw_temperatura",
    "distancia_max_idw_temperatura_km"
]


COLUNAS_NAO_ALTERADAS = [
    coluna
    for coluna in inmet_integrado_correcao.columns
    if coluna not in COLUNAS_CORRIGIDAS
]


original_validacao = (
    inmet_integrado_correcao
    .sort_values(
        [
            "codigo_ibge",
            "ano"
        ]
    )
    .reset_index(
        drop=True
    )
    [
        COLUNAS_NAO_ALTERADAS
    ]
)


corrigido_validacao = (
    inmet_integrado_corrigido
    .sort_values(
        [
            "codigo_ibge",
            "ano"
        ]
    )
    .reset_index(
        drop=True
    )
    [
        COLUNAS_NAO_ALTERADAS
    ]
)


pd.testing.assert_frame_equal(
    original_validacao,
    corrigido_validacao,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12
)


print(
    "OK — todas as demais colunas permaneceram inalteradas."
)    # ============================================================
# GARANTIA — SOMENTE DUAS COLUNAS FORAM CORRIGIDAS
# ============================================================

COLUNAS_CORRIGIDAS = [
    "numero_estacoes_idw_temperatura",
    "distancia_max_idw_temperatura_km"
]


COLUNAS_NAO_ALTERADAS = [
    coluna
    for coluna in inmet_integrado_correcao.columns
    if coluna not in COLUNAS_CORRIGIDAS
]


original_validacao = (
    inmet_integrado_correcao
    .sort_values(
        [
            "codigo_ibge",
            "ano"
        ]
    )
    .reset_index(
        drop=True
    )
    [
        COLUNAS_NAO_ALTERADAS
    ]
)


corrigido_validacao = (
    inmet_integrado_corrigido
    .sort_values(
        [
            "codigo_ibge",
            "ano"
        ]
    )
    .reset_index(
        drop=True
    )
    [
        COLUNAS_NAO_ALTERADAS
    ]
)


pd.testing.assert_frame_equal(
    original_validacao,
    corrigido_validacao,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12
)


print(
    "OK — todas as demais colunas permaneceram inalteradas."
)

OK — todas as demais colunas permaneceram inalteradas.
OK — todas as demais colunas permaneceram inalteradas.


In [208]:
# ============================================================
# BACKUP DA VERSÃO ANTERIOR
# ============================================================

QUALITY_INMET_DIR = (
    BASE_DIR
    / "data"
    / "databases_processed"
    / "inmet"
    / "quality"
)


QUALITY_INMET_DIR.mkdir(
    parents=True,
    exist_ok=True
)


ARQUIVO_BACKUP_INMET = (
    QUALITY_INMET_DIR
    / "inmet_municipio_ano_2019_2024_antes_correcao_idw_temperatura.csv"
)


inmet_integrado_correcao.to_csv(
    ARQUIVO_BACKUP_INMET,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Backup salvo:"
)

print(
    ARQUIVO_BACKUP_INMET
)

Backup salvo:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_processed\inmet\quality\inmet_municipio_ano_2019_2024_antes_correcao_idw_temperatura.csv


In [209]:
# ============================================================
# ATUALIZAÇÃO DA CURATED INMET
# ============================================================

inmet_integrado_corrigido.to_csv(
    ARQUIVO_INMET_INTEGRADO,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Curated INMET atualizada:"
)

print(
    ARQUIVO_INMET_INTEGRADO
)

Curated INMET atualizada:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\inmet\municipio_ano\inmet_municipio_ano_2019_2024.csv


In [210]:
# ============================================================
# VALIDAÇÃO FINAL — ARQUIVO CORRIGIDO EM DISCO
# ============================================================

inmet_check_final = pd.read_csv(
    ARQUIVO_INMET_INTEGRADO,
    dtype={
        "codigo_ibge": "string"
    }
)


inmet_check_final[
    "codigo_ibge"
] = (
    inmet_check_final[
        "codigo_ibge"
    ]
    .str.zfill(7)
)


print(
    "Dimensão:"
)

print(
    inmet_check_final.shape
)


print(
    "\nDuplicatas codigo_ibge + ano:"
)

print(
    inmet_check_final
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


print(
    "\nNulos das variáveis climáticas principais:"
)

display(
    inmet_check_final[
        [
            "precipitacao_anual_mm",
            "temperatura_media_anual_c",
            "umidade_media_anual_pct"
        ]
    ]
    .isna()
    .sum()
)


print(
    "\nMetadados IDW de temperatura por origem:"
)

display(
    inmet_check_final
    .groupby(
        "origem_temperatura"
    )
    [
        [
            "numero_estacoes_idw_temperatura",
            "distancia_max_idw_temperatura_km"
        ]
    ]
    .count()
)

Dimensão:
(9966, 30)

Duplicatas codigo_ibge + ano:
0

Nulos das variáveis climáticas principais:


precipitacao_anual_mm        0
temperatura_media_anual_c    0
umidade_media_anual_pct      0
dtype: int64


Metadados IDW de temperatura por origem:


,numero_estacoes_idw_temperatura,distancia_max_idw_temperatura_km
origem_temperatura,,
estimado_idw,9191,9191
observado_inmet,0,0
